# Federated Transfer Learning Across Heterogeneous Medical Imaging Domains

## Aim

**To determine whether federated transfer learning can build a useful shared model across five
medical imaging domains that have nothing visually in common, and to measure exactly what that
sharing costs and what it buys.**

The five domains are a chest X-ray, a breast ultrasound, a dermatoscopic skin photograph, a
histopathology slide and a fundus photograph of the retina. Each one lives at a separate client
and never leaves it. This is the hard case for federated learning: most published work assumes
clients whose data differs statistically but comes from the same modality, whereas here the
modalities themselves differ.

## What this notebook demonstrates

Every experiment below exists to answer one stated question. The plan is given in the next cell
and each section repeats its own question before the code runs. Nothing is reported that was not
produced by a run in this notebook; where a run failed or where an earlier result turned out to
be caused by a defect in this code, that is stated in place rather than removed.

## Two label spaces, one protocol

The study is run twice over the same five clients.

- **Superset (`union`)**, 25 classes. Every client keeps its own native classes and they are
  concatenated into one shared output space of 2+2+7+9+5 = 25. This is the general-purpose
  multi-disease model: one network that can name any condition of any modality.
- **Binary (`binary`)**, 2 classes. Every client is collapsed to healthy vs diseased, so all five
  share one common 2-class output space.

The binary setting is not a simplification for convenience. It is the only setting in which a
model pretrained on four clients has a head that is *meaningful* for the fifth, which makes a
genuine zero-shot bootstrap test possible. The superset setting cannot support that test, because
the held-out client's block of the output layer never received a gradient.

## Relationship to published work

None of the algorithms are new. FedAvg (McMahan et al., 2017), FedProx (Li et al., 2020),
FedAdam/FedOpt (Reddi et al., 2021), Federated Distillation / FedDF (Jeong et al., 2018; Lin et
al., 2020), local BatchNorm / FedBN (Li et al., 2021), shared-backbone-with-personalized-head
(Collins et al., 2021), mixture-of-experts routing (Shazeer et al., 2017; Fedus et al., 2022),
FedMix (Yoon et al., 2021), Grad-CAM (Selvaraju et al., 2017) and CKA (Kornblith et al., 2019)
are all established. The datasets are MedMNIST v2 (Yang et al., 2023). The contribution here is
the experimental study: this particular combination of five disjoint modalities, the measurements,
and the conclusions drawn from them.

## Running this notebook

Set the runtime to a GPU (Runtime -> Change runtime type -> T4 GPU), then Runtime -> Run all.
Results are written to Google Drive after every individual experiment and reloaded when the
notebook starts, so an interrupted session is resumed simply by running it again: every stage
checks what is already recorded and skips it.

The full campaign is about four and a half hours of GPU time. Free Colab sessions are usually
shorter than that, so expect to run the notebook two or three times. That is the intended way to
use it, not a workaround.

If only part of the campaign is wanted, switch the stages off in the `STAGES` dictionary in
Section 2. The export in the last section works with whatever has been recorded so far: stages
that have not run are left out of the report rather than blocking it.


---
## Experiment plan

Each row states what is to be demonstrated, the experiment that demonstrates it, and where the
result appears. The stage column maps to the `STAGES` switches in the configuration cell, so the
notebook can be executed in parts.

| # | Question to answer | Experiment | Primary measure | Section | Stage |
|---|---|---|---|---|---|
| Q1 | Does federation beat training alone, and does it reach centralized training? | Centralized, local-only and four federated algorithms on the same clients | accuracy, macro F1, AUC per client | 10, 17 | `core` |
| Q2 | Is the comparison fair when centralized sees S images and each client sees S/n? | Fix total passes E and sweep every (local epochs e, rounds R) with e·R = E | accuracy at matched sample exposure | 19 | `fair` |
| Q3 | Does label granularity change the answer? | Repeat with 2, 3, 4 and all native classes per client | centralized-minus-federated gap | 11 | `core` |
| Q4 | Can a mixture of experts absorb modality heterogeneity, and how many experts are useful? | FedMoE with K = 2, 3, 4, 6, 7, 10 at top-1, plus top-2 routing at K = 6, 10, plus an experts-only ablation; FedMix | accuracy, macro F1, AUC, experts actually used, gate entropy | 20 | `moe` |
| Q5 | Can a new client bootstrap from the federation, and does it beat plain ImageNet? | Leave-one-client-out: pretrain on four, transfer to the fifth at 0, 10, 25, 50, 100 % of its data, against an ImageNet control | accuracy vs data fraction, per held-out client | 21 | `bootstrap` |
| Q6 | What does a bootstrapped model achieve with **zero** local data? | The 0 % point of Q5, with three arms: backbone+head transplanted, backbone only, ImageNet; scored against the majority-class baseline | accuracy vs majority baseline, prediction spread | 22 | `bootstrap` |
| Q7 | Do the answers hold when all five clients share one healthy/diseased label space? | The entire campaign of Q1, Q4, Q5 and Q6 repeated in `binary` mode | same measures, compared against the superset run | 28, 29 | `binary` |
| Q8 | What does the privacy mechanism cost? | Measure the true update norm, then sweep the clipping bound and the noise level | accuracy vs clipping and noise | 13 | `privacy` |
| Q9 | How does the method behave as clients are added? | Run FedAvg on subsets of 2, 3, 4 and 5 clients | accuracy, MB, time vs number of clients | 23 | `extras` |
| Q10 | How much of any difference is random? | Repeat FedAvg over three seeds | per-client standard deviation | 24 | `extras` |
| Q11 | Did the shared backbone learn anything transferable at all? | CKA between clients, linear probe on frozen features against ImageNet, Grad-CAM | probe accuracy, CKA | 16 | `extras` |

Two rules apply to the whole report and are stated once here.

1. **The noise floor.** Section 24 measures the per-client standard deviation across seeds. Any
   difference smaller than that floor is reported as "no difference", never as a ranking.
2. **Every method carries accuracy, macro F1 and AUC.** Accuracy alone is misleading on these
   datasets: DermaMNIST has a 58:1 class imbalance, so a model that predicts only the majority
   class scores well on accuracy and badly on macro F1.


---
## 1. Setup

In [ ]:
import sys
!{sys.executable} -m pip install -q torch torchvision medmnist scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.8 MB/s eta 0:00:00


In [ ]:
import os, time, json, copy, random, itertools
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset, ConcatDataset
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, roc_auc_score)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
import warnings; warnings.filterwarnings('ignore')

from medmnist import (PneumoniaMNIST, BreastMNIST, DermaMNIST, PathMNIST,
                      RetinaMNIST, BloodMNIST)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

print('Device :', DEVICE, '| PyTorch', torch.__version__)
if DEVICE.type == 'cpu':
    print('Running on CPU. Enable a GPU runtime before running the full grid.')

Device : cuda | PyTorch 2.11.0+cu128


---
## 2. Configuration

`label_space` controls the whole experiment:

- `binary` reproduces the original setup (every client collapsed to normal/abnormal).
- `native` gives every client its own native classes and a **local** classification head; only the
  backbone is federated. This is the personalized setting.
- `union` concatenates the five label spaces into one shared output space (2+2+7+9+5 = 25 classes)
  and federates the head as well. This is the general-purpose multidisease model: one network, one
  output layer, every disease state of every modality.

`classes_per_client` restricts each client to its k most frequent native classes, which is how the
3-class and 4-class experiments are produced. `None` keeps all native classes.

`RUN` switches individual experiments on and off so the notebook can be executed in stages.

In [ ]:
CONFIG = {
    'num_rounds'          : 8,
    'local_epochs'        : 4,
    'batch_size'          : 64,
    'learning_rate'       : 0.0005,
    'image_size'          : 96,
    'mu_prox'             : 0.01,
    'server_lr'           : 0.1,
    'beta1'               : 0.9,
    'beta2'               : 0.99,
    'tau'                 : 1e-3,
    'max_train_per_client': 2000,
    'max_val_per_client'  : 500,
    'max_test_per_client' : 1000,
    'label_space'         : 'union',
    'classes_per_client'  : None,
    'num_public'          : 250,
    'distill_epochs'      : 8,
    'temperature'         : 4.0,
    'seeds'               : [0],
    'new_disease'         : 'BloodMNIST',
    'new_disease_epochs'  : 10,
    'new_disease_train'   : 1500,
    # Q4: how many experts. Five clients with top-1 routing can occupy at most five experts, so
    # the counts above five are the test of whether extra capacity is used or wasted.
    'expert_counts'       : [2, 3, 4, 6, 7, 10],
    'expert_counts_top2'  : [6, 10],
    'expert_counts_binary': [3, 6, 10],
    # Q5/Q6: fractions of the held-out client's own training data. 0.0 is the zero-shot point.
    'bootstrap_fractions' : [0.0, 0.1, 0.25, 0.5, 1.0],
    'bootstrap_algo'      : 'fedadam',
}

# Stages. Each one is roughly self-contained and writes its results to disk when it finishes, so
# the notebook can be run in parts and resumed with load_results() after a disconnect.
#
#   core      ~55 min   Q1, Q3   baselines, four federated algorithms, granularity sweep
#   fair      ~27 min   Q2       matched sample exposure
#   privacy   ~30 min   Q8       clipping and noise
#   moe       ~40 min   Q4       expert sweep in the superset label space
#   bootstrap ~45 min   Q5, Q6   leave-one-client-out in the superset label space
#   binary    ~85 min   Q7       the whole campaign again in healthy/diseased
#   extras    ~35 min   Q9-Q11   scalability, seeds, representation, explainability
#   export     ~1 min            tables.tex, figures and results files for the report
STAGES = {
    'core'     : True,
    'fair'     : True,
    'privacy'  : True,
    'moe'      : True,
    'bootstrap': True,
    'binary'   : True,
    'extras'   : True,
    'export'   : True,
}

RUN = {
    'centralized'      : STAGES['core'],
    'local_only'       : STAGES['core'],
    'federated'        : STAGES['core'],
    'distillation'     : STAGES['core'],
    'multiclass_sweep' : STAGES['core'],
    'distill_ablation' : STAGES['core'],
    'privacy'          : STAGES['privacy'],
    'new_disease'      : STAGES['bootstrap'],
    'explainability'   : STAGES['extras'],
    'representation'   : STAGES['extras'],
    'fair_protocol'    : STAGES['fair'],
    'moe_mix'          : STAGES['moe'],
    'bootstrap'        : STAGES['bootstrap'],
    'zero_shot_ablation': STAGES['bootstrap'],
    'scalability'      : STAGES['extras'],
    'reliability'      : STAGES['extras'],
    'binary_campaign'  : STAGES['binary'],
}

BINARY_MAPS = {
    'PneumoniaMNIST' : {0: 0, 1: 1},
    'BreastMNIST'    : {0: 1, 1: 0},
    'DermaMNIST'     : {0: 1, 1: 1, 2: 0, 3: 0, 4: 1, 5: 0, 6: 0},
    'PathMNIST'      : {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 1, 8: 1},
    'RetinaMNIST'    : {0: 0, 1: 1, 2: 1, 3: 1, 4: 1},
}

DATASET_LOADERS = {
    'PneumoniaMNIST' : PneumoniaMNIST,
    'BreastMNIST'    : BreastMNIST,
    'DermaMNIST'     : DermaMNIST,
    'PathMNIST'      : PathMNIST,
    'RetinaMNIST'    : RetinaMNIST,
}

MODALITY = {
    'PneumoniaMNIST' : 'chest X-ray',
    'BreastMNIST'    : 'breast ultrasound',
    'DermaMNIST'     : 'dermatoscopy',
    'PathMNIST'      : 'histopathology',
    'RetinaMNIST'    : 'fundus photography',
}

RESULTS = []
HISTORY = []

def record(store, **kw):
    store.append(kw)
    return kw

def _fmt(v, w=7, nd=4):
    """Format a metric that may legitimately be missing. AUC is undefined for a client whose
    test split does not contain every one of its classes, so it can be None."""
    return f'{v:{w}.{nd}f}' if isinstance(v, (int, float)) else f'{"-":>{w}}'

def tag_rows(start_index, **kw):
    """Label every RESULTS row appended since start_index. Used to mark which campaign a row
    belongs to without changing the signature of every run_* function."""
    for r in RESULTS[start_index:]:
        r.update(kw)


---
## 2b. Where results are stored

Every experiment writes its rows to Google Drive the moment it finishes, and the notebook reloads
them when it starts. Three things follow from that, and they are the reason this cell exists at
all rather than at the end of the notebook:

- A disconnected runtime, an exhausted GPU quota or a closed tab costs at most the one experiment
  that was in flight. Everything before it is already on Drive.
- Re-running the notebook does not repeat finished work. Each driver checks `RESULTS` first and
  skips anything already recorded, so "Run all" after a crash continues instead of starting over.
- The file is written to a temporary name and then renamed, so an interruption during the write
  leaves the previous good copy intact rather than a half-written one.

This must run **before** any training cell. A save function defined below the training cells
would not be in effect while they run, and the results would go to the container's own disk,
which is erased when the runtime restarts.


In [ ]:
import gc
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/fl_thesis'
RESULTS_FILE = os.path.join(SAVE_DIR, 'results.json')
os.makedirs(SAVE_DIR, exist_ok=True)

def save_results(path=None):
    tmp = RESULTS_FILE + '.tmp'
    with open(tmp, 'w') as f:
        json.dump({'config': CONFIG, 'results': RESULTS, 'history': HISTORY}, f,
                  indent=1, default=str)
    os.replace(tmp, RESULTS_FILE)
    print(f'  [saved {len(RESULTS)} rows to Drive]')

def load_results(path=None):
    """Pull previously saved rows back into memory. Rows already present are not duplicated, so
    this is safe to call more than once."""
    if not os.path.exists(RESULTS_FILE) or os.path.getsize(RESULTS_FILE) == 0:
        print('no previous results on Drive - starting fresh')
        return
    try:
        blob = json.load(open(RESULTS_FILE))
    except json.JSONDecodeError:
        print('saved file is unreadable - starting fresh')
        return
    # The key has to separate every run that can legitimately share a setting. The
    # matched-exposure sweep, for instance, writes six FedAvg runs under setting='fair_protocol'
    # that differ only in local_epochs and fair_rounds; without those fields five of the six are
    # discarded as duplicates on reload.
    key = lambda r: (r.get('experiment'), r.get('method'), r.get('client'), r.get('seed'),
                     r.get('setting'), r.get('fraction'), r.get('num_experts'), r.get('init'),
                     r.get('local_epochs'), r.get('fair_rounds'), r.get('top_k'),
                     r.get('n_public'), r.get('sync_clients'), r.get('dp_clip'),
                     r.get('dp_noise'), r.get('classes_per_client'))
    have = {key(r) for r in RESULTS}
    added = 0
    for r in blob.get('results', []):
        if key(r) not in have:
            RESULTS.append(r)
            have.add(key(r))
            added += 1
    for h in blob.get('history', []):
        HISTORY.append(h)
    print(f'loaded {added} rows from Drive (total {len(RESULTS)})')

def already_done(**crit):
    """True when a run matching these fields is already recorded. Every guard in the notebook
    goes through this, so a re-run continues rather than repeating finished work."""
    return any(all(r.get(k) == v for k, v in crit.items()) for r in RESULTS)

def gpu_report(tag=''):
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f'  [{tag} GPU free {free / 1e9:.2f} GB of {total / 1e9:.2f} GB]')

def release(obj, depth=0):
    """Drop the model objects a finished run still holds and hand the memory back to the GPU.
    Each run keeps five client models and a global one with their optimizer state, roughly 0.8 GB;
    a dozen of them exhausts a 15 GB card. The metrics are already in RESULTS, so nothing measured
    is lost."""
    if isinstance(obj, dict) and depth <= 3:
        if {'locals', 'global', 'model'} & set(obj):
            for k in ('locals', 'global', 'model'):
                obj.pop(k, None)
        else:
            for v in list(obj.values()):
                release(v, depth + 1)
    if depth == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def checkpoint(start, campaign=None, run=None):
    """Called after every individual experiment: label the new rows, write them to Drive, and
    free the weights."""
    if campaign:
        for r in RESULTS[start:]:
            r['campaign'] = campaign
    save_results()
    if run is not None:
        release(run)
    return len(RESULTS)

load_results()
print('GPU available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again.')


Mounted at /content/drive
no previous results on Drive - starting fresh
GPU available: True


---
## 3. Data

Same preprocessing as before. Two changes that matter:

- the MedMNIST **validation** split is now loaded and used as the reference/public pool for
  Federated Distillation. In the previous version that pool was taken from the *test* sets, which
  leaks test data into training.
- the label map is derived once from the training split and reused for val and test, so the class
  indices are consistent.

In [ ]:
imagenet_mean, imagenet_std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
transform_test = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

class MappedDataset(Dataset):
    def __init__(self, base, indices, label_map, offset=0):
        self.base = base
        self.indices = indices
        self.label_map = label_map
        self.offset = offset
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, lab = self.base[self.indices[i]]
        native = int(np.array(lab).reshape(-1)[0])
        return img, self.label_map[native] + self.offset

def dataset_labels(ds):
    return np.array(ds.labels).reshape(-1)

def make_label_map(name, y, label_space, k):
    if label_space == 'binary':
        return BINARY_MAPS[name]
    classes = sorted(np.unique(y).tolist())
    if k is not None:
        counts = {c: int((y == c).sum()) for c in classes}
        classes = sorted(sorted(classes, key=lambda c: -counts[c])[:k])
    return {c: i for i, c in enumerate(classes)}

def build_split(loader_class, split, transform, cap, label_map, seed):
    base = loader_class(split=split, transform=transform, download=True)
    y = dataset_labels(base)
    keep = np.where(np.isin(y, list(label_map)))[0]
    if cap is not None and len(keep) > cap:
        keep = np.random.RandomState(seed).choice(keep, cap, replace=False)
    return MappedDataset(base, keep.tolist(), label_map)

def make_clients(label_space=None, classes_per_client=None, seed=0):
    label_space = label_space or CONFIG['label_space']
    if classes_per_client is None:
        classes_per_client = CONFIG['classes_per_client']
    names = list(DATASET_LOADERS)
    train, val, test, num_classes = [], [], [], []
    for name in names:
        loader_class = DATASET_LOADERS[name]
        probe = loader_class(split='train', download=True)
        label_map = make_label_map(name, dataset_labels(probe), label_space, classes_per_client)
        train.append(build_split(loader_class, 'train', transform_train,
                                 CONFIG['max_train_per_client'], label_map, seed))
        val.append(build_split(loader_class, 'val', transform_test,
                               CONFIG['max_val_per_client'], label_map, seed))
        test.append(build_split(loader_class, 'test', transform_test,
                                CONFIG['max_test_per_client'], label_map, seed))
        num_classes.append(len(set(label_map.values())))

    offsets = [0] * len(names)
    if label_space == 'union':
        offsets = np.cumsum([0] + num_classes[:-1]).tolist()
        for off, tr, va, te in zip(offsets, train, val, test):
            tr.offset = va.offset = te.offset = off

    class_ids = [list(range(o, o + n)) for o, n in zip(offsets, num_classes)]
    total_classes = sum(num_classes) if label_space == 'union' else max(num_classes)

    return {
        'names'        : names,
        'train'        : train,
        'val'          : val,
        'test'         : test,
        'num_classes'  : num_classes,
        'class_ids'    : class_ids,
        'total_classes': total_classes,
        'label_space'  : label_space,
        'sizes'        : [len(d) for d in train],
        'train_loaders': [DataLoader(d, batch_size=CONFIG['batch_size'], shuffle=True) for d in train],
        'test_loaders' : [DataLoader(d, batch_size=128, shuffle=False) for d in test],
    }

def head_size(clients, i):
    return clients['total_classes'] if clients['label_space'] == 'union' else clients['num_classes'][i]

def label_counts(ds, num_classes):
    y = dataset_labels(ds.base)[ds.indices]
    mapped = np.array([ds.label_map[int(v)] for v in y])
    return np.bincount(mapped, minlength=num_classes)

def describe(clients):
    rows = []
    for i, name in enumerate(clients['names']):
        counts = label_counts(clients['train'][i], clients['num_classes'][i])
        rows.append({
            'client': name,
            'modality': MODALITY[name],
            'classes': clients['num_classes'][i],
            'train': len(clients['train'][i]),
            'val': len(clients['val'][i]),
            'test': len(clients['test'][i]),
            'imbalance': round(float(counts.max() / max(counts.min(), 1)), 2),
        })
    return rows

CLIENTS = make_clients()
TABLE1 = describe(CLIENTS)
for r in TABLE1:
    print(r)
print('total classes in shared output space:', CLIENTS['total_classes'])

100%|██████████| 4.17M/4.17M [00:06<00:00, 604kB/s]
100%|██████████| 560k/560k [00:01<00:00, 509kB/s]
100%|██████████| 19.7M/19.7M [00:14<00:00, 1.39MB/s]
100%|██████████| 206M/206M [01:02<00:00, 3.28MB/s]
100%|██████████| 3.29M/3.29M [00:04<00:00, 753kB/s] 


{'client': 'PneumoniaMNIST', 'modality': 'chest X-ray', 'classes': 2, 'train': 2000, 'val': 500, 'test': 624, 'imbalance': 2.8}
{'client': 'BreastMNIST', 'modality': 'breast ultrasound', 'classes': 2, 'train': 546, 'val': 78, 'test': 156, 'imbalance': 2.71}
{'client': 'DermaMNIST', 'modality': 'dermatoscopy', 'classes': 7, 'train': 2000, 'val': 500, 'test': 1000, 'imbalance': 58.52}
{'client': 'PathMNIST', 'modality': 'histopathology', 'classes': 9, 'train': 2000, 'val': 500, 'test': 1000, 'imbalance': 1.61}
{'client': 'RetinaMNIST', 'modality': 'fundus photography', 'classes': 5, 'train': 1080, 'val': 120, 'test': 400, 'imbalance': 7.36}
total classes in shared output space: 25


---
## 4. Evaluation

Accuracy alone is not informative on these datasets (PneumoniaMNIST and PathMNIST are strongly
imbalanced, and in `union` mode a client's classes are a small slice of the output space). Each
evaluation returns accuracy, macro precision/recall/F1, per-class recall, the confusion matrix and
an AUC where one is mathematically defined.

In `union` mode two accuracies are reported:

- **within-client accuracy**: argmax restricted to the client's own block of classes, i.e. the
  clinically realistic case where the modality of the incoming image is known.
- **open accuracy**: argmax over all 25 logits, plus `outside_rate`, the fraction of test images
  assigned to a class belonging to a different modality. `outside_rate` is the measurement that
  tells us whether the general-purpose head is actually usable without knowing the modality.

In [ ]:
def collect_predictions(model, loader, device=DEVICE):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            out = model(imgs.to(device))
            ps.append(F.softmax(out, dim=1).cpu().numpy())
            ys.append(labels.numpy())
    return np.concatenate(ys), np.concatenate(ps)

def safe_auc(y_local, p_local):
    n = p_local.shape[1]
    try:
        if n == 2:
            return float(roc_auc_score(y_local, p_local[:, 1]))
        if len(np.unique(y_local)) < n:
            return None
        return float(roc_auc_score(y_local, p_local, multi_class='ovr',
                                   average='macro', labels=list(range(n))))
    except ValueError:
        return None

def compute_metrics(y, prob, class_ids):
    ids = np.array(class_ids)
    pred = ids[prob[:, ids].argmax(1)]
    acc = accuracy_score(y, pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, labels=ids,
                                                    average='macro', zero_division=0)
    _, per_class_recall, _, support = precision_recall_fscore_support(
        y, pred, labels=ids, average=None, zero_division=0)
    p_local = prob[:, ids]
    p_local = p_local / p_local.sum(1, keepdims=True).clip(1e-12)
    y_local = np.searchsorted(ids, y)
    out = {
        'accuracy'  : float(acc),
        'precision' : float(pr),
        'recall'    : float(rc),
        'f1'        : float(f1),
        'auc'       : safe_auc(y_local, p_local),
        'per_class_recall': [round(float(v), 4) for v in per_class_recall],
        'support'   : [int(v) for v in support],
        'confusion' : confusion_matrix(y, pred, labels=ids).tolist(),
    }
    if prob.shape[1] > len(ids):
        open_pred = prob.argmax(1)
        out['open_accuracy'] = float((open_pred == y).mean())
        out['outside_rate'] = float((~np.isin(open_pred, ids)).mean())
    return out

def evaluate_client(model, loader, class_ids):
    y, prob = collect_predictions(model, loader)
    return compute_metrics(y, prob, class_ids)

def mean_over_clients(metric_list, key):
    vals = [m[key] for m in metric_list if m.get(key) is not None]
    return float(np.mean(vals)) if vals else None

---
## 5. Model

Unchanged from the previous version except that the head width is now a parameter and the backbone
can be exposed separately for Grad-CAM and for representation analysis. The stem (`conv1`, `bn1`)
stays frozen; everything else is fine-tuned.

Which parameters are federated depends on the mode:

| mode | backbone conv | BatchNorm affine + running stats | head |
|---|---|---|---|
| `binary` / `union` | federated | **local** (FedBN) | federated |
| `native` | federated | **local** | **local** (per-client heads) |

BatchNorm is never federated in any mode, following FedBN (Li et al., 2021) — the earlier collapse
to constant predictions is exactly the failure mode that paper describes for multi-source medical
imaging.

In [ ]:
class FederatedModel(nn.Module):
    def __init__(self, num_classes, pretrained=True, freeze_stem=True):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet18(weights=weights)
        backbone.fc = nn.Identity()
        self.backbone = backbone
        if freeze_stem:
            for p in self.backbone.conv1.parameters():
                p.requires_grad = False
            for p in self.backbone.bn1.parameters():
                p.requires_grad = False
        self.head = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    def features(self, x):
        return self.backbone(x)
    def forward(self, x):
        return self.head(self.backbone(x))

def bn_param_names(model):
    names = set()
    for name, module in model.named_modules():
        if isinstance(module, nn.BatchNorm2d):
            names.add(name + '.weight')
            names.add(name + '.bias')
    return names

def federated_names(model, local_head=False):
    bn = bn_param_names(model)
    out = []
    for n, p in model.named_parameters():
        if not p.requires_grad or n in bn:
            continue
        if local_head and n.startswith('head.'):
            continue
        out.append(n)
    return out

def extract_state(model, names):
    params = dict(model.named_parameters())
    return {n: params[n].detach().clone() for n in names}

def load_state(model, state):
    full = model.state_dict()
    for k, v in state.items():
        full[k] = v
    model.load_state_dict(full)

def state_bytes(state):
    return sum(v.numel() * v.element_size() for v in state.values())

_probe = FederatedModel(CLIENTS['total_classes'])
print('total parameters     :', f'{sum(p.numel() for p in _probe.parameters()):,}')
print('trainable            :', f'{sum(p.numel() for p in _probe.parameters() if p.requires_grad):,}')
print('federated (union)    :', f'{sum(_probe.state_dict()[n].numel() for n in federated_names(_probe)):,}')
print('federated (native)   :', f'{sum(_probe.state_dict()[n].numel() for n in federated_names(_probe, local_head=True)):,}')
print('kept local (BN)      :', f'{sum(_probe.state_dict()[n].numel() for n in bn_param_names(_probe)):,}')
del _probe

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 166MB/s]


total parameters     : 11,245,401
trainable            : 11,235,865
federated (union)    : 11,226,393
federated (native)   : 11,157,504
kept local (BN)      : 9,600


---
## 6. Local training, differential privacy mechanism, communication accounting

`train_local` covers FedAvg/FedAdam (no proximal term), FedProx (`mu > 0`) and the centralized and
local-only baselines.

`privatize` implements the update-level Gaussian mechanism used in the privacy experiment: the
client update is clipped to an L2 norm `C` and Gaussian noise with standard deviation
`noise_multiplier * C` is added before it leaves the client. **This is the mechanism, not a formal
guarantee.** With 5 fixed clients, every round, and no subsampling, there is no meaningful
(epsilon, delta) to report — see the discussion in Section 14. The experiment measures the
accuracy cost of the noise, and nothing stronger is claimed.

Communication is counted from the tensors the code actually moves: `state_bytes` on the exact
dictionaries that are sent up and broadcast down.

In [ ]:
def train_local(model, loader, optimizer, epochs, global_params=None, mu=0.0, device=DEVICE):
    model.train()
    ce = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = ce(model(imgs), labels)
            if global_params is not None and mu > 0:
                prox = sum(((p - global_params[n]) ** 2).sum()
                           for n, p in model.named_parameters() if n in global_params)
                loss = loss + (mu / 2) * prox
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

CLIP_NORMS = []

def privatize(model, global_state, clip_norm, noise_multiplier, device=DEVICE):
    params = dict(model.named_parameters())
    delta = {n: params[n].detach() - global_state[n] for n in global_state}
    total = torch.sqrt(sum((d.float() ** 2).sum() for d in delta.values()))
    scale = min(1.0, clip_norm / (total.item() + 1e-12))
    new_state = {}
    for n, d in delta.items():
        d = d * scale
        if noise_multiplier > 0:
            d = d + torch.normal(0.0, noise_multiplier * clip_norm, size=d.shape, device=device)
        new_state[n] = global_state[n] + d
    load_state(model, new_state)
    CLIP_NORMS.append(float(total.item()))
    return float(total.item())

class Comm:
    def __init__(self):
        self.up = 0
        self.down = 0
    def send(self, nbytes, n=1):
        self.up += nbytes * n
    def receive(self, nbytes, n=1):
        self.down += nbytes * n
    def total_mb(self):
        return (self.up + self.down) / 1e6
    def summary(self, rounds, n_clients):
        return {
            'up_mb': self.up / 1e6,
            'down_mb': self.down / 1e6,
            'total_mb': self.total_mb(),
            'mb_per_client_per_round': self.total_mb() / max(rounds * n_clients, 1),
            'rounds': rounds,
        }

---
## 7. Baselines: centralized and local-only

Two reference points are needed to say anything about federated learning.

**Centralized** pools all five training sets into one dataset and trains one model with the same
architecture, preprocessing, splits and label space. It is the upper bound that federation gives up.
It requires a shared output space, so it runs in `binary` or `union` mode. One caveat is recorded
honestly rather than hidden: the centralized model sees mixed-modality batches, so its BatchNorm
layers are shared across modalities, whereas the federated clients keep BatchNorm local. That is a
real difference between the two settings, not an experimental artefact that can be removed — a
centralized model has no client identity to personalize on. A second centralized variant with
per-modality BatchNorm would be a different (and non-deployable) model.

**Local-only** trains each client alone on its own data for the same total number of epochs
(`num_rounds * local_epochs`), with no communication at all. This is the baseline that answers "why
federate": if federation does not beat local-only, there is no reason to pay the communication and
privacy cost.

Epoch budgets are matched per client. The centralized model necessarily performs more gradient steps
per epoch because its dataset is five times larger; wall-clock time is reported alongside so the
comparison is visible rather than assumed away.

In [ ]:
def run_centralized(clients, seed=0, epochs=None, setting='main'):
    set_seed(seed)
    epochs = epochs or CONFIG['num_rounds'] * CONFIG['local_epochs']
    pooled = ConcatDataset(clients['train'])
    loader = DataLoader(pooled, batch_size=CONFIG['batch_size'], shuffle=True)
    model = FederatedModel(clients['total_classes']).to(DEVICE)
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG['learning_rate'])
    t0 = time.time()
    train_local(model, loader, opt, epochs)
    elapsed = time.time() - t0
    per_client = [evaluate_client(model, ld, ids)
                  for ld, ids in zip(clients['test_loaders'], clients['class_ids'])]
    for i, (name, m) in enumerate(zip(clients['names'], per_client)):
        record(RESULTS, experiment='centralized', method='Centralized', client=name, seed=seed,
               label_space=clients['label_space'], classes=clients['num_classes'][i], setting=setting,
               time_s=elapsed, up_mb=0.0, down_mb=0.0, rounds=0, **{k: v for k, v in m.items()
               if k not in ('confusion', 'per_class_recall', 'support')})
    print(f'centralized | {epochs} epochs | {elapsed:.0f}s | mean acc '
          f'{mean_over_clients(per_client, "accuracy"):.4f} | mean F1 {mean_over_clients(per_client, "f1"):.4f}')
    return {'model': model, 'metrics': per_client, 'time_s': elapsed}

def run_local_only(clients, seed=0, epochs=None, setting='main'):
    set_seed(seed)
    epochs = epochs or CONFIG['num_rounds'] * CONFIG['local_epochs']
    out = []
    t0 = time.time()
    for i, name in enumerate(clients['names']):
        model = FederatedModel(head_size(clients, i)).to(DEVICE)
        opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG['learning_rate'])
        train_local(model, clients['train_loaders'][i], opt, epochs)
        m = evaluate_client(model, clients['test_loaders'][i], clients['class_ids'][i])
        out.append(m)
        record(RESULTS, experiment='local_only', method='Local-only', client=name, seed=seed,
               label_space=clients['label_space'], classes=clients['num_classes'][i],
               setting=setting, time_s=time.time() - t0, up_mb=0.0, down_mb=0.0, rounds=0,
               **{k: v for k, v in m.items() if k not in ('confusion', 'per_class_recall', 'support')})
        print(f'  local-only {name:16s} acc {m["accuracy"]:.4f} f1 {m["f1"]:.4f}')
    print(f'local-only | mean acc {mean_over_clients(out, "accuracy"):.4f} | '
          f'mean F1 {mean_over_clients(out, "f1"):.4f}')
    return {'metrics': out}

---
## 8. Federated algorithms

FedAvg, FedProx and FedAdam share one loop; the only differences are the proximal term during local
training and the server-side update rule. Each client is evaluated with its own local BatchNorm and,
in `native` mode, its own head — this is a personalized federated model, and the evaluation
protocol is stated explicitly rather than left implicit.

Optionally a differential-privacy mechanism is applied to every client update before aggregation.

In [ ]:
def aggregate(states, sizes):
    total = sum(sizes)
    keys = list(states[0])
    return {k: sum(states[i][k].float() * (sizes[i] / total) for i in range(len(states))) for k in keys}

def run_federated(clients, algo='fedavg', seed=0, rounds=None, dp=None, setting='main', verbose=True):
    set_seed(seed)
    rounds = rounds or CONFIG['num_rounds']
    local_head = clients['label_space'] == 'native'
    n = len(clients['names'])

    locals_ = [FederatedModel(head_size(clients, i)).to(DEVICE) for i in range(n)]
    opts = [optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=CONFIG['learning_rate'])
            for m in locals_]
    global_model = FederatedModel(head_size(clients, 0)).to(DEVICE)
    fed_names = federated_names(global_model, local_head=local_head)
    comm = Comm()
    server_m, server_v = {}, {}
    label = {'fedavg': 'FedAvg', 'fedprox': 'FedProx', 'fedadam': 'FedAdam'}[algo]
    if dp is not None:
        label += f" (DP C={dp['clip']} s={dp['noise']})"

    t0 = time.time()
    for rnd in range(1, rounds + 1):
        g_state = extract_state(global_model, fed_names)
        nbytes = state_bytes(g_state)
        for m in locals_:
            load_state(m, g_state)
            comm.receive(nbytes)

        for i, m in enumerate(locals_):
            train_local(m, clients['train_loaders'][i], opts[i], CONFIG['local_epochs'],
                        global_params=g_state if algo == 'fedprox' else None,
                        mu=CONFIG['mu_prox'] if algo == 'fedprox' else 0.0)
            if dp is not None:
                privatize(m, g_state, dp['clip'], dp['noise'])

        client_states = [extract_state(m, fed_names) for m in locals_]
        for s in client_states:
            comm.send(state_bytes(s))
        avg = aggregate(client_states, clients['sizes'])

        if algo == 'fedadam':
            new_state = {}
            for k in avg:
                delta = avg[k] - g_state[k].float()
                if k not in server_m:
                    server_m[k] = torch.zeros_like(delta)
                    server_v[k] = torch.full_like(delta, CONFIG['tau'] ** 2)
                server_m[k] = CONFIG['beta1'] * server_m[k] + (1 - CONFIG['beta1']) * delta
                server_v[k] = CONFIG['beta2'] * server_v[k] + (1 - CONFIG['beta2']) * delta ** 2
                new_state[k] = (g_state[k].float() + CONFIG['server_lr'] * server_m[k]
                                / (torch.sqrt(server_v[k]) + CONFIG['tau']))
            load_state(global_model, new_state)
        else:
            load_state(global_model, avg)

        metrics = [evaluate_client(m, ld, ids)
                   for m, ld, ids in zip(locals_, clients['test_loaders'], clients['class_ids'])]
        for name, mt in zip(clients['names'], metrics):
            record(HISTORY, method=label, algo=algo, seed=seed, round=rnd, client=name,
                   label_space=clients['label_space'], accuracy=mt['accuracy'], f1=mt['f1'],
                   cum_mb=comm.total_mb(), elapsed_s=time.time() - t0)
        if verbose:
            accs = ' | '.join(f'{nm[:5]} {mt["accuracy"]:.3f}' for nm, mt in zip(clients['names'], metrics))
            print(f'  {label} r{rnd:02d} | mean acc {mean_over_clients(metrics, "accuracy"):.4f} | '
                  f'mean f1 {mean_over_clients(metrics, "f1"):.4f} | {accs}')

    elapsed = time.time() - t0
    summary = comm.summary(rounds, n)
    for i, (name, mt) in enumerate(zip(clients['names'], metrics)):
        record(RESULTS, experiment='federated' if dp is None else 'privacy', method=label, algo=algo,
               client=name, seed=seed, label_space=clients['label_space'],
               classes=clients['num_classes'][i], setting=setting, time_s=elapsed,
               dp_clip=None if dp is None else dp['clip'],
               dp_noise=None if dp is None else dp['noise'],
               up_mb=summary['up_mb'], down_mb=summary['down_mb'], rounds=rounds,
               **{k: v for k, v in mt.items() if k not in ('confusion', 'per_class_recall', 'support')})
    print(f'{label} | mean acc {mean_over_clients(metrics, "accuracy"):.4f} | '
          f'mean F1 {mean_over_clients(metrics, "f1"):.4f} | {summary["total_mb"]:.1f} MB | {elapsed:.0f}s')
    return {'locals': locals_, 'global': global_model, 'metrics': metrics,
            'comm': summary, 'time_s': elapsed, 'label': label}

---
## 9. Federated Distillation

The previous implementation of FedDF scored far below the weight-averaging methods. Four problems in
that implementation are corrected here, and they are listed explicitly because the corrected result
is the point of the experiment, not the original number:

1. **The reference pool was drawn from the test sets.** It now comes from the MedMNIST validation
   split.
2. **The ensemble targets were softmaxed twice.** The client outputs were already probabilities, and
   `softmax(probabilities / T)` flattens a distribution over 25 classes to something very close to
   uniform, so the KL loss carried almost no signal. Client *logits* are now averaged and the
   temperature is applied once.
3. **The server optimizer was rebuilt every round**, discarding Adam's moments each time.
4. **The distilled weights overwrote the clients completely**, so a model fitted on 250 reference
   images replaced models fitted on 2000 local images each round. The clients now keep their local
   BatchNorm (as before) and the overwrite is optional via `sync_clients`.

Communication is accounted differently from the weight-averaging methods: the uplink carries a
`num_public x num_classes` matrix of logits instead of the full parameter set. The downlink still
carries the distilled model, because FedDF broadcasts a model — so this variant is cheap on the
uplink only. That asymmetry is measured, not assumed.

In [ ]:
def build_public_pool(clients, num_public, seed=0):
    per_client = max(num_public // len(clients['names']), 1)
    rng = np.random.RandomState(seed)
    parts = []
    for ds in clients['val']:
        idx = rng.choice(len(ds), min(per_client, len(ds)), replace=False)
        parts.append(Subset(ds, idx.tolist()))
    pool = ConcatDataset(parts)
    loader = DataLoader(pool, batch_size=128, shuffle=False)
    X = torch.cat([b[0] for b in loader])
    return X

def client_logits(model, X, device=DEVICE, batch=128):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            out.append(model(X[i:i + batch].to(device)).cpu())
    return torch.cat(out)

def run_distillation(clients, seed=0, rounds=None, num_public=None, sync_clients=True,
                     setting='main', verbose=True):
    set_seed(seed)
    rounds = rounds or CONFIG['num_rounds']
    num_public = num_public or CONFIG['num_public']
    if clients['label_space'] == 'native':
        raise ValueError('distillation needs a shared output space')
    n = len(clients['names'])
    X = build_public_pool(clients, num_public, seed)

    locals_ = [FederatedModel(head_size(clients, i)).to(DEVICE) for i in range(n)]
    opts = [optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=CONFIG['learning_rate'])
            for m in locals_]
    global_model = FederatedModel(head_size(clients, 0)).to(DEVICE)
    fed_names = federated_names(global_model)
    global_opt = optim.Adam(filter(lambda p: p.requires_grad, global_model.parameters()),
                            lr=CONFIG['learning_rate'])
    comm = Comm()
    weights = np.array(clients['sizes']) / sum(clients['sizes'])
    T = CONFIG['temperature']
    label = (f'FedDistill (n_public={len(X)})' if sync_clients
             else f'Local-only (distill server only, n_public={len(X)})')

    t0 = time.time()
    for rnd in range(1, rounds + 1):
        for i, m in enumerate(locals_):
            train_local(m, clients['train_loaders'][i], opts[i], CONFIG['local_epochs'])

        logits = [client_logits(m, X) for m in locals_]
        for lg in logits:
            comm.send(lg.numel() * lg.element_size())
        ensemble = sum(w * lg for w, lg in zip(weights, logits)).to(DEVICE)
        target = F.softmax(ensemble / T, dim=1)

        global_model.train()
        for _ in range(CONFIG['distill_epochs']):
            perm = torch.randperm(len(X))
            for i in range(0, len(X), CONFIG['batch_size']):
                idx = perm[i:i + CONFIG['batch_size']]
                global_opt.zero_grad()
                out = global_model(X[idx].to(DEVICE))
                loss = F.kl_div(F.log_softmax(out / T, dim=1), target[idx],
                                reduction='batchmean') * T ** 2
                loss.backward()
                nn.utils.clip_grad_norm_(global_model.parameters(), max_norm=1.0)
                global_opt.step()

        if sync_clients:
            distilled = extract_state(global_model, fed_names)
            nbytes = state_bytes(distilled)
            for m in locals_:
                load_state(m, distilled)
                comm.receive(nbytes)

        metrics = [evaluate_client(m, ld, ids)
                   for m, ld, ids in zip(locals_, clients['test_loaders'], clients['class_ids'])]
        for name, mt in zip(clients['names'], metrics):
            record(HISTORY, method=label, algo='feddistill', seed=seed, round=rnd, client=name,
                   label_space=clients['label_space'], accuracy=mt['accuracy'], f1=mt['f1'],
                   cum_mb=comm.total_mb(), elapsed_s=time.time() - t0)
        if verbose:
            print(f'  {label} r{rnd:02d} | mean acc {mean_over_clients(metrics, "accuracy"):.4f} | '
                  f'mean f1 {mean_over_clients(metrics, "f1"):.4f}')

    elapsed = time.time() - t0
    summary = comm.summary(rounds, n)
    for i, (name, mt) in enumerate(zip(clients['names'], metrics)):
        record(RESULTS, experiment='federated', method=label, algo='feddistill', client=name,
               seed=seed, label_space=clients['label_space'], classes=clients['num_classes'][i],
               n_public=len(X), sync_clients=sync_clients, setting=setting, time_s=elapsed,
               up_mb=summary['up_mb'], down_mb=summary['down_mb'], rounds=rounds,
               **{k: v for k, v in mt.items() if k not in ('confusion', 'per_class_recall', 'support')})
    global_metrics = [evaluate_client(global_model, ld, ids)
                      for ld, ids in zip(clients['test_loaders'], clients['class_ids'])]
    for i, (name, mt) in enumerate(zip(clients['names'], global_metrics)):
        record(RESULTS, experiment='distill_global', method=f'FedDistill server model (n_public={len(X)})',
               algo='feddistill_global', client=name, seed=seed, label_space=clients['label_space'],
               classes=clients['num_classes'][i], n_public=len(X), sync_clients=sync_clients,
               setting=setting, rounds=rounds,
               **{k: v for k, v in mt.items() if k not in ('confusion', 'per_class_recall', 'support')})
    print(f'{label} | clients: acc {mean_over_clients(metrics, "accuracy"):.4f} '
          f'f1 {mean_over_clients(metrics, "f1"):.4f} | server model: acc '
          f'{mean_over_clients(global_metrics, "accuracy"):.4f} | {summary["total_mb"]:.1f} MB')
    return {'locals': locals_, 'global_metrics': global_metrics, 'global': global_model, 'metrics': metrics,
            'comm': summary, 'time_s': elapsed, 'label': label}

---
## 10. Main run

Baselines and the four federated methods, in the label space set in `CONFIG`. Run over several seeds
by extending `CONFIG['seeds']` — the tables at the end aggregate mean and standard deviation over
whatever seeds are present, and say so when only one seed was run.

In [ ]:
# MODELS keeps one trained model per method for the Grad-CAM, representation and new-disease
# sections. Those are the only places weights are needed after a run, so everything else is freed
# as it goes.
MODELS = {}

for seed in CONFIG['seeds']:
    _s = len(RESULTS)
    if RUN['centralized']:
        if already_done(experiment='centralized', setting='main', seed=seed):
            print(f'centralized seed={seed} already done, skipping')
        else:
            MODELS[('centralized', seed)] = run_centralized(CLIENTS, seed=seed)
            _s = checkpoint(_s, campaign='union')
    if RUN['local_only']:
        if already_done(experiment='local_only', setting='main', seed=seed):
            print(f'local-only seed={seed} already done, skipping')
        else:
            run_local_only(CLIENTS, seed=seed)
            _s = checkpoint(_s, campaign='union')
    if RUN['federated']:
        for algo in ['fedavg', 'fedprox', 'fedadam']:
            if already_done(experiment='federated', algo=algo, setting='main', seed=seed):
                print(f'{algo} seed={seed} already done, skipping')
                continue
            MODELS[(algo, seed)] = run_federated(CLIENTS, algo=algo, seed=seed)
            _s = checkpoint(_s, campaign='union')
    if RUN['distillation']:
        if already_done(experiment='federated', algo='feddistill', setting='main', seed=seed):
            print(f'distillation seed={seed} already done, skipping')
        else:
            MODELS[('feddistill', seed)] = run_distillation(CLIENTS, seed=seed)
            _s = checkpoint(_s, campaign='union')

if not MODELS:
    print('no models in memory (all runs were skipped). Sections 15, 16 and 21 need them, so '
          'they will be skipped too; delete those rows from Drive and re-run if you need them.')

centralized | 32 epochs | 203s | mean acc 0.7209 | mean F1 0.5710
  [saved 5 rows to Drive]
  local-only PneumoniaMNIST   acc 0.8494 f1 0.8216
  local-only BreastMNIST      acc 0.8654 f1 0.8349
  local-only DermaMNIST       acc 0.6700 f1 0.3211
  local-only PathMNIST        acc 0.8040 f1 0.7625
  local-only RetinaMNIST      acc 0.3800 f1 0.2725
local-only | mean acc 0.7137 | mean F1 0.6025
  [saved 10 rows to Drive]
  FedAvg r01 | mean acc 0.7220 | mean f1 0.5428 | Pneum 0.939 | Breas 0.763 | Derma 0.687 | PathM 0.776 | Retin 0.445
  FedAvg r02 | mean acc 0.7245 | mean f1 0.5829 | Pneum 0.755 | Breas 0.865 | Derma 0.696 | PathM 0.764 | Retin 0.542
  FedAvg r03 | mean acc 0.7256 | mean f1 0.5666 | Pneum 0.907 | Breas 0.814 | Derma 0.673 | PathM 0.739 | Retin 0.495
  FedAvg r04 | mean acc 0.7244 | mean f1 0.5720 | Pneum 0.888 | Breas 0.782 | Derma 0.709 | PathM 0.768 | Retin 0.475
  FedAvg r05 | mean acc 0.6899 | mean f1 0.5789 | Pneum 0.867 | Breas 0.801 | Derma 0.601 | PathM 0.745 | Re

---
## 11. Multiclass sweep

The original experiment collapsed every client to normal/abnormal. That choice is convenient (one
shared head) but it discards most of the label information in DermaMNIST, PathMNIST and RetinaMNIST,
and it makes the task artificially easy for the imbalanced clients. This sweep re-runs the same
pipeline at 2, 3 and 4 classes per client (each client's most frequent native classes) and with the
full native label sets, so the effect of label granularity on the federated/centralized gap can be
measured instead of assumed.

FedAvg and the centralized baseline are run at every setting; the other algorithms are left out of
the sweep to keep the runtime manageable, since the sweep is about label granularity rather than
about the choice of aggregation rule.

In [ ]:
if RUN['multiclass_sweep']:
    for k in [2, 3, 4, None]:
        tag = f'sweep_k{k}'
        if already_done(setting=tag):
            print(f'granularity k={k} already done, skipping')
            continue
        cl = make_clients(label_space='union', classes_per_client=k, seed=0)
        print(f'--- classes_per_client={k} | per-client classes {cl["num_classes"]} '
              f'| output space {cl["total_classes"]} ---')
        for r in describe(cl):
            print('   ', r)
        _s = len(RESULTS)
        release(run_centralized(cl, seed=0, setting=tag))
        release(run_federated(cl, algo='fedavg', seed=0, setting=tag, verbose=False))
        # The sweep rows carry their own setting so they never mix with the main campaign in the
        # master table, and classes_per_client is recorded so the table can be grouped by it.
        for r in RESULTS[_s:]:
            r['classes_per_client'] = k if k is not None else 'all'
        checkpoint(_s, campaign='granularity')
else:
    print('multiclass sweep not run')

--- classes_per_client=2 | per-client classes [2, 2, 2, 2, 2] | output space 10 ---
    {'client': 'PneumoniaMNIST', 'modality': 'chest X-ray', 'classes': 2, 'train': 2000, 'val': 500, 'test': 624, 'imbalance': 2.8}
    {'client': 'BreastMNIST', 'modality': 'breast ultrasound', 'classes': 2, 'train': 546, 'val': 78, 'test': 156, 'imbalance': 2.71}
    {'client': 'DermaMNIST', 'modality': 'dermatoscopy', 'classes': 2, 'train': 2000, 'val': 500, 'test': 1000, 'imbalance': 5.85}
    {'client': 'PathMNIST', 'modality': 'histopathology', 'classes': 2, 'train': 2000, 'val': 500, 'test': 1000, 'imbalance': 1.12}
    {'client': 'RetinaMNIST', 'modality': 'fundus photography', 'classes': 2, 'train': 692, 'val': 82, 'test': 266, 'imbalance': 2.36}
centralized | 32 epochs | 187s | mean acc 0.8576 | mean F1 0.7868
FedAvg | mean acc 0.8212 | mean F1 0.7749 | 3591.8 MB | 205s
  [saved 45 rows to Drive]
--- classes_per_client=3 | per-client classes [2, 2, 3, 3, 3] | output space 13 ---
    {'client':

---
## 12. Federated Distillation: reference-set size ablation

If the gap between FedDF and the weight-averaging methods comes from the amount of knowledge that
can be carried by a small reference set, then accuracy should increase with the size of that set
while the uplink cost grows linearly with it. If the gap persists at 500 reference images, the
bottleneck is elsewhere (heterogeneity of the ensemble targets across modalities, or the fact that
a distilled model fitted on out-of-client data replaces well-fitted local models).

Both `sync_clients=True` (clients are overwritten by the distilled model, as in the original
implementation) and `sync_clients=False` (clients keep their own weights, the distilled model is
only the server-side artefact) are run at one size, because that switch isolates the fourth problem
listed in Section 9.

In [ ]:
if RUN['distill_ablation']:
    if already_done(setting='ablation'):
        print('distillation ablation already done, skipping')
    else:
        _s = len(RESULTS)
        for n_pub in [50, 100, 250, 500]:
            release(run_distillation(CLIENTS, seed=0, num_public=n_pub, setting='ablation',
                                     verbose=False))
            _s = checkpoint(_s, campaign='distill_ablation')
        # With sync_clients=False the clients receive nothing from the server, so those five
        # models are local-only models; only the server model is a federated artefact.
        release(run_distillation(CLIENTS, seed=0, num_public=250, sync_clients=False,
                                 setting='ablation', verbose=False))
        checkpoint(_s, campaign='distill_ablation')
else:
    print('distillation ablation not run')

FedDistill (n_public=50) | clients: acc 0.5146 f1 0.2155 | server model: acc 0.5474 | 1796.4 MB
  [saved 85 rows to Drive]


KeyboardInterrupt: 

---
## 13. Privacy

Client updates are clipped and Gaussian noise is added before aggregation. Three noise levels plus
the clipping-only control are run, so the accuracy cost of the mechanism can be separated from the
cost of clipping alone.

What is **not** claimed: this is not an (epsilon, delta)-differentially private training run. A
formal guarantee would require per-example gradient clipping with a subsampled Gaussian mechanism
and an RDP/PLD accountant (as in Abadi et al., 2016, or Opacus), and with five fixed clients
participating in every round the amplification-by-subsampling that makes those bounds usable does
not apply. The numbers below measure the utility cost of the noise mechanism. A properly accounted
DP-SGD run is listed as future work in Section 18.

In [ ]:
# C = 1.0 clips an 11M-parameter update to almost nothing, so the earlier sweep measured clipping
# collapse rather than the cost of noise. The real update norm is measured first, and the clipping
# grid is placed around it.
if RUN['privacy'] and already_done(experiment='privacy'):
    print('privacy sweep already done, skipping')
elif RUN['privacy']:
    CLIP_NORMS.clear()
    run_federated(CLIENTS, algo='fedavg', seed=0, rounds=1, dp={'clip': 1e9, 'noise': 0.0},
                  setting='norm_probe', verbose=False)
    observed = np.array(CLIP_NORMS)
    med = float(np.median(observed))
    print(f'observed client update L2 norms (1 round, no clipping): '
          f'min {observed.min():.2f} median {med:.2f} max {observed.max():.2f}')
    RESULTS[:] = [r for r in RESULTS if r.get('setting') != 'norm_probe']
    HISTORY[:] = [h for h in HISTORY if 'C=1000000000' not in str(h.get('method', ''))]

    clip_grid = sorted({1.0, round(med / 4, 2), round(med / 2, 2), round(med, 2)})
    print('clipping grid:', clip_grid)
    _s = len(RESULTS)
    for clip in clip_grid:
        release(run_federated(CLIENTS, algo='fedavg', seed=0, dp={'clip': clip, 'noise': 0.0},
                              setting='privacy', verbose=False))
        _s = checkpoint(_s, campaign='privacy')

    clip_rows = [r for r in RESULTS if r.get('setting') == 'privacy' and r.get('dp_noise') == 0.0]
    best_clip = max(clip_grid, key=lambda c: mean_over_clients(
        [r for r in clip_rows if r['dp_clip'] == c], 'accuracy') or 0.0)
    print(f'clipping level carried into the noise sweep: C={best_clip}')
    for noise in [0.001, 0.01, 0.05]:
        release(run_federated(CLIENTS, algo='fedavg', seed=0,
                              dp={'clip': best_clip, 'noise': noise},
                              setting='privacy', verbose=False))
        _s = checkpoint(_s, campaign='privacy')
else:
    print('privacy experiment not run')


---
## 14. Bootstrapping a new disease

A sixth dataset that took no part in federated training is introduced as a new client. Three
initializations are compared at 10%, 25%, 50% and 100% of its training data:

1. **scratch** — randomly initialized ResNet18.
2. **ImageNet** — the standard transfer-learning baseline.
3. **federated** — convolutional weights taken from the federated global model.

The ImageNet arm is the control that makes this experiment interpretable. Without it, any gain from
the federated initialization could simply be the ImageNet pretraining that the federated model also
started from. The question is specifically whether federated fine-tuning on five medical domains
adds anything *on top of* ImageNet features for a sixth, unseen domain.

BatchNorm parameters were never federated, so they start from ImageNet values in both the ImageNet
and federated arms; the only difference between those two arms is the convolutional weights.

Terminology, kept separate on purpose: *transfer learning* is reusing learned weights as an
initialization (all three arms differ only in initialization); *federated learning* is the
collaborative training procedure that produced one of those initializations without any client
sharing raw images. Federated Distillation is a federated aggregation method, not transfer learning.

In [ ]:
def build_new_disease(name, seed=0):
    loader_class = {'BloodMNIST': BloodMNIST}[name]
    probe = loader_class(split='train', download=True)
    label_map = make_label_map(name, dataset_labels(probe), 'native', None)
    train = build_split(loader_class, 'train', transform_train, CONFIG['new_disease_train'], label_map, seed)
    test = build_split(loader_class, 'test', transform_test, CONFIG['max_test_per_client'], label_map, seed)
    return train, test, len(set(label_map.values()))

def finetune_new_disease(init, train_ds, test_ds, num_classes, fraction, seed=0, source_model=None):
    set_seed(seed)
    idx = np.random.RandomState(seed).choice(len(train_ds), max(int(len(train_ds) * fraction), 16),
                                             replace=False)
    loader = DataLoader(Subset(train_ds, idx.tolist()), batch_size=CONFIG['batch_size'], shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

    model = FederatedModel(num_classes, pretrained=(init != 'scratch')).to(DEVICE)
    if init == 'federated':
        src = source_model.state_dict()
        dst = model.state_dict()
        for k in dst:
            if k.startswith('backbone.') and 'bn' not in k and 'downsample.1' not in k and k in src:
                dst[k] = src[k].clone()
        model.load_state_dict(dst)
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG['learning_rate'])
    t0 = time.time()
    train_local(model, loader, opt, CONFIG['new_disease_epochs'])
    m = evaluate_client(model, test_loader, list(range(num_classes)))
    record(RESULTS, experiment='new_disease', method=init, client=CONFIG['new_disease'], seed=seed,
           fraction=fraction, n_train=len(idx), classes=num_classes, time_s=time.time() - t0,
           **{k: v for k, v in m.items() if k not in ('confusion', 'per_class_recall', 'support')})
    print(f'  {init:10s} frac {fraction:<5} n={len(idx):<5} acc {m["accuracy"]:.4f} f1 {m["f1"]:.4f}')
    return m

if RUN['new_disease'] and ('fedavg', CONFIG['seeds'][0]) in MODELS:
    nd_train, nd_test, nd_classes = build_new_disease(CONFIG['new_disease'])
    source = MODELS[('fedavg', CONFIG['seeds'][0])]['global']
    print(f'{CONFIG["new_disease"]}: {len(nd_train)} train / {len(nd_test)} test / {nd_classes} classes')
    for frac in [0.1, 0.25, 0.5, 1.0]:
        for init in ['scratch', 'imagenet', 'federated']:
            finetune_new_disease(init, nd_train, nd_test, nd_classes, frac, seed=CONFIG['seeds'][0],
                                 source_model=source)
else:
    print('new-disease experiment not run')

---
## 15. Explainability

Grad-CAM on the last convolutional block, for the same fixed images under each method. Two things
are produced: the qualitative panel, and a quantitative agreement measure — the IoU between the
top-20% activated regions of two methods on the same image. High agreement means the aggregation
rule does not change where the model looks; low agreement means the choice of federated algorithm
changes model behaviour in a way that accuracy alone does not reveal.

Heatmaps are used here to inspect model behaviour. They are not evidence of clinical validity, and
no radiological or histological claim is made from them. Grad-CAM is known to be sensitive to the
chosen layer and to produce plausible-looking maps even for wrong predictions.

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model = model
        self.activations = None
        self.gradients = None
        layer.register_forward_hook(self._forward)
        layer.register_full_backward_hook(self._backward)
    def _forward(self, module, inp, out):
        self.activations = out.detach()
    def _backward(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()
    def __call__(self, x, target=None):
        self.model.eval()
        self.model.zero_grad()
        out = self.model(x)
        if target is None:
            target = out.argmax(1)
        out.gather(1, target.view(-1, 1)).sum().backward()
        w = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.activations).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False).squeeze(1)
        raw = cam.cpu().numpy()
        lo = cam.amin(dim=(1, 2), keepdim=True)
        hi = cam.amax(dim=(1, 2), keepdim=True)
        norm = ((cam - lo) / (hi - lo + 1e-8)).cpu().numpy()
        return norm, raw, target.cpu().numpy()

def unnormalize(img):
    x = img.permute(1, 2, 0).cpu().numpy() * np.array(imagenet_std) + np.array(imagenet_mean)
    return np.clip(x, 0, 1)

def cam_iou(a, b, q=0.8):
    # a CAM whose ReLU killed every pixel normalizes to a constant; a >= quantile then selects
    # the whole image and the IoU is an artefact (1.0 for two dead maps, 0.2 for one). Those
    # comparisons are dropped instead of being reported as agreement.
    if a.max() <= 0 or b.max() <= 0:
        return None
    ma, mb = a > np.quantile(a, q), b > np.quantile(b, q)
    if ma.sum() == 0 or mb.sum() == 0:
        return None
    return float((ma & mb).sum() / (ma | mb).sum())

if RUN['explainability'] and MODELS:
    xai_methods = {}
    for key in [('centralized', CONFIG['seeds'][0]), ('fedavg', CONFIG['seeds'][0]),
                ('fedprox', CONFIG['seeds'][0]), ('fedadam', CONFIG['seeds'][0]),
                ('feddistill', CONFIG['seeds'][0])]:
        if key in MODELS:
            xai_methods[key[0]] = MODELS[key]

    samples = []
    for i, name in enumerate(CLIENTS['names']):
        img, lab = CLIENTS['test'][i][0]
        samples.append((name, img, int(lab)))

    cams, dead_maps = {}, {}
    for method, bundle in xai_methods.items():
        cams[method], dead_maps[method] = [], 0
        explainers = {}
        for i, (name, img, lab) in enumerate(samples):
            model = bundle['model'] if method == 'centralized' else bundle['locals'][i]
            if id(model) not in explainers:
                explainers[id(model)] = GradCAM(model, model.backbone.layer4)
            target = torch.tensor([lab], device=DEVICE)
            norm, raw, _ = explainers[id(model)](img.unsqueeze(0).to(DEVICE), target=target)
            if raw[0].max() <= 0:
                dead_maps[method] += 1
            cams[method].append(raw[0])
    print('degenerate (all-zero) CAMs per method, out of', len(samples), ':', dead_maps)

    fig, axes = plt.subplots(len(samples), len(cams) + 1, figsize=(2.4 * (len(cams) + 1), 2.4 * len(samples)))
    for r, (name, img, lab) in enumerate(samples):
        axes[r, 0].imshow(unnormalize(img))
        axes[r, 0].set_ylabel(name, fontsize=8)
        axes[r, 0].set_title('input' if r == 0 else '', fontsize=9)
        axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
        for c, method in enumerate(cams, start=1):
            cam = cams[method][r]
            axes[r, c].imshow(unnormalize(img))
            if cam.max() > 0:
                axes[r, c].imshow(cam / cam.max(), cmap='jet', alpha=0.45)
            else:
                axes[r, c].set_title('dead CAM', fontsize=7, color='red')
            if r == 0:
                axes[r, c].set_title(method, fontsize=9)
            axes[r, c].axis('off')
    plt.tight_layout()
    plt.savefig('gradcam_panel.png', dpi=140, bbox_inches='tight')
    plt.show()

    method_list = list(cams)
    print()
    print('Grad-CAM top-20% region IoU, target = true label, degenerate maps excluded:')
    for a, b in itertools.combinations(method_list, 2):
        ious = [cam_iou(cams[a][i], cams[b][i]) for i in range(len(samples))]
        ok = [v for v in ious if v is not None]
        if not ok:
            print(f'  {a:14s} vs {b:14s} : no valid pair (all maps degenerate)')
            continue
        record(RESULTS, experiment='explainability', method=f'{a} vs {b}', metric='cam_iou',
               value=float(np.mean(ok)), n_valid=len(ok), n_total=len(ious),
               per_client=[None if v is None else round(v, 3) for v in ious])
        print(f'  {a:14s} vs {b:14s} : {np.mean(ok):.3f}   (valid {len(ok)}/{len(ious)})')
else:
    print('explainability not run')


---
## 16. Shared representation across modalities

Three measurements, aimed at the question of whether federated training over five unrelated imaging
domains produces a representation that carries information across them, or whether each client ends
up with a private solution that happens to share a parameter vector.

1. **Linear CKA** between the penultimate features of the five client models on one common probe
   batch containing images from all modalities. High CKA means the clients encode the probe images
   similarly despite their local BatchNorm.
2. **t-SNE / PCA** of the federated features, coloured by client. If the embedding separates cleanly
   by modality, the representation is dominated by modality identity rather than by pathology.
3. **Linear probe transfer**: freeze the backbone, fit a logistic regression on each client's
   features, and compare the federated backbone against the plain ImageNet backbone. This is the
   quantitative version of "is the shared representation useful" — if the federated features give a
   better linear probe than ImageNet features on the same data, federated training added something.

"Common root cause" is interpreted here strictly as shared *representational* structure. Nothing in
this analysis can support a claim about a shared biological cause of these diseases, and none is made.

In [ ]:
def linear_cka(X, Y):
    X = X - X.mean(0, keepdims=True)
    Y = Y - Y.mean(0, keepdims=True)
    num = np.linalg.norm(Y.T @ X, 'fro') ** 2
    den = np.linalg.norm(X.T @ X, 'fro') * np.linalg.norm(Y.T @ Y, 'fro')
    return float(num / (den + 1e-12))

def features_of(model, X, batch=128):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            out.append(model.features(X[i:i + batch].to(DEVICE)).cpu().numpy())
    return np.concatenate(out)

def stack_split(ds, n, seed):
    idx = np.random.RandomState(seed).choice(len(ds), min(n, len(ds)), replace=False)
    X = torch.stack([ds[int(j)][0] for j in idx])
    y = np.array([ds[int(j)][1] for j in idx])
    return X, y

if RUN['representation'] and ('fedavg', CONFIG['seeds'][0]) in MODELS:
    probe_imgs, probe_client = [], []
    for i, name in enumerate(CLIENTS['names']):
        idx = np.random.RandomState(0).choice(len(CLIENTS['test'][i]), 60, replace=False)
        for j in idx:
            img, _ = CLIENTS['test'][i][int(j)]
            probe_imgs.append(img)
            probe_client.append(i)
    probe_X = torch.stack(probe_imgs)
    probe_client = np.array(probe_client)

    fed = MODELS[('fedavg', CONFIG['seeds'][0])]
    feats = [features_of(m, probe_X) for m in fed['locals']]

    cka = np.zeros((len(feats), len(feats)))
    for a in range(len(feats)):
        for b in range(len(feats)):
            cka[a, b] = linear_cka(feats[a], feats[b])
    off = cka[~np.eye(len(feats), dtype=bool)]
    record(RESULTS, experiment='representation', method='FedAvg', metric='mean_pairwise_cka',
           value=float(off.mean()), matrix=cka.round(3).tolist())
    print('Linear CKA between client backbones (FedAvg, shared probe batch):')
    print('        ' + ' '.join(f'{n[:6]:>7s}' for n in CLIENTS['names']))
    for i, n in enumerate(CLIENTS['names']):
        print(f'{n[:7]:8s}' + ' '.join(f'{v:7.3f}' for v in cka[i]))
    print(f'mean off-diagonal CKA: {off.mean():.3f}')

    emb = TSNE(n_components=2, init='pca', perplexity=25, random_state=0).fit_transform(
        PCA(n_components=50, random_state=0).fit_transform(features_of(fed['global'], probe_X)))
    plt.figure(figsize=(6, 5))
    for i, name in enumerate(CLIENTS['names']):
        sel = probe_client == i
        plt.scatter(emb[sel, 0], emb[sel, 1], s=12, label=name)
    plt.legend(fontsize=7)
    plt.title('t-SNE of federated backbone features, coloured by client')
    plt.tight_layout()
    plt.savefig('representation_tsne.png', dpi=140, bbox_inches='tight')
    plt.show()

    # The probe is fitted on the validation split and scored on the test split. Fitting it on the
    # training split would let the federated backbone score its own training images, which the
    # ImageNet backbone has never seen; that comparison measures memorization, not representation.
    print()
    print('Linear probe on frozen features (fit on val split, scored on test split):')
    imagenet_backbone = FederatedModel(2).to(DEVICE)
    for i, name in enumerate(CLIENTS['names']):
        Xtr, ytr = stack_split(CLIENTS['val'][i], 400, 0)
        Xte, yte = stack_split(CLIENTS['test'][i], 400, 1)
        for tag, backbone in [('imagenet', imagenet_backbone), ('federated', fed['locals'][i])]:
            Ztr, Zte = features_of(backbone, Xtr), features_of(backbone, Xte)
            clf = LogisticRegression(max_iter=2000).fit(Ztr, ytr)
            acc = float(clf.score(Zte, yte))
            record(RESULTS, experiment='linear_probe', method=tag, client=name, accuracy=acc,
                   n_fit=len(ytr), n_eval=len(yte))
            print(f'  {name:16s} {tag:10s} probe acc {acc:.4f}')
else:
    print('representation analysis not run')


---
## 17. Tables and figures

Everything below is assembled from `RESULTS` and `HISTORY`. If an experiment was not run its table
prints "no records" instead of a number.

In [ ]:
def rows(**filters):
    out = []
    for r in RESULTS:
        if all(r.get(k) == v for k, v in filters.items()):
            out.append(r)
    return out

def agg(records, group_keys, value_keys):
    groups = {}
    for r in records:
        key = tuple(r.get(k) for k in group_keys)
        groups.setdefault(key, []).append(r)
    out = []
    for key, rs in groups.items():
        row = dict(zip(group_keys, key))
        for v in value_keys:
            vals = [r[v] for r in rs if r.get(v) is not None and not isinstance(r.get(v), (list, dict))]
            row[v] = (float(np.mean(vals)), float(np.std(vals)), len(vals)) if vals else None
        out.append(row)
    return out

def show(title, records, group_keys, value_keys, sort_by=None):
    print('\n' + title)
    if not records:
        print('  no records - experiment not run')
        return
    table = agg(records, group_keys, value_keys)
    if sort_by:
        table.sort(key=lambda r: (r[sort_by][0] if r.get(sort_by) else -1), reverse=True)
    width = {k: max(len(str(k)), max(len(str(r[k])) for r in table)) for k in group_keys}
    header = '  ' + ' | '.join(f'{k:{width[k]}s}' for k in group_keys) + ' | ' + \
             ' | '.join(f'{v:>16s}' for v in value_keys)
    print(header)
    print('  ' + '-' * (len(header) - 2))
    for r in table:
        cells = []
        for v in value_keys:
            if r[v] is None:
                cells.append(f'{"-":>16s}')
            elif r[v][2] > 1:
                cells.append(f'{r[v][0]:.4f}+-{r[v][1]:.3f}'.rjust(16))
            else:
                cells.append(f'{r[v][0]:.4f}'.rjust(16))
        print('  ' + ' | '.join(f'{str(r[k]):{width[k]}s}' for k in group_keys) + ' | ' + ' | '.join(cells))

print('TABLE 1 - dataset characteristics')
for r in TABLE1:
    print('  ', r)

main = [r for r in RESULTS if r['experiment'] in ('centralized', 'local_only', 'federated')
        and r.get('setting') == 'main']

show('TABLE 2/3 - centralized, local-only and federated, averaged over clients and seeds',
     main, ['method'], ['accuracy', 'f1', 'recall', 'auc', 'outside_rate'], sort_by='accuracy')

show('TABLE 3b - per client', main, ['method', 'client'], ['accuracy', 'f1', 'auc'])

show('TABLE 4 - communication and time',
     [r for r in main if r['experiment'] == 'federated'],
     ['method'], ['accuracy', 'f1', 'up_mb', 'down_mb', 'time_s'])

show('TABLE 5 - privacy mechanism (clipping + Gaussian noise on client updates; no formal epsilon)',
     rows(experiment='privacy'), ['dp_clip', 'dp_noise'], ['accuracy', 'f1', 'auc'])

show('TABLE 6 - new-disease transfer',
     rows(experiment='new_disease'), ['method', 'fraction'], ['accuracy', 'f1', 'auc'])

show('TABLE 7a - Grad-CAM agreement between methods',
     rows(experiment='explainability'), ['method'], ['value'])

show('TABLE 7b - linear probe on frozen features',
     rows(experiment='linear_probe'), ['method', 'client'], ['accuracy'])

show('TABLE 8 - label granularity sweep',
     [r for r in RESULTS if r.get('setting') in ('main', 'sweep')
      and r['experiment'] in ('centralized', 'federated') and r.get('classes')],
     ['method', 'classes'], ['accuracy', 'f1'])

show('TABLE 9 - Federated Distillation reference-set ablation (client models)',
     [r for r in RESULTS if r.get('algo') == 'feddistill'],
     ['n_public', 'sync_clients'], ['accuracy', 'f1', 'up_mb', 'down_mb'])

show('TABLE 9b - Federated Distillation server model',
     rows(experiment='distill_global'), ['n_public', 'sync_clients'], ['accuracy', 'f1'])

with open('results.json', 'w') as f:
    json.dump({'config': CONFIG, 'table1': TABLE1, 'results': RESULTS, 'history': HISTORY},
              f, indent=1, default=str)
print('\nsaved results.json')

In [ ]:
def history_curve(method, key='accuracy'):
    rs = [r for r in HISTORY if r['method'] == method]
    if not rs:
        return None, None
    rounds = sorted(set(r['round'] for r in rs))
    return rounds, [float(np.mean([r[key] for r in rs if r['round'] == x])) for x in rounds]

methods = sorted(set(r['method'] for r in HISTORY))
if methods:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for m in methods:
        x, y = history_curve(m, 'accuracy')
        axes[0].plot(x, y, '-o', ms=3, label=m)
        x, y = history_curve(m, 'f1')
        axes[1].plot(x, y, '-o', ms=3, label=m)
        rs = [r for r in HISTORY if r['method'] == m]
        rounds = sorted(set(r['round'] for r in rs))
        mb = [float(np.mean([r['cum_mb'] for r in rs if r['round'] == x])) for x in rounds]
        acc = [float(np.mean([r['accuracy'] for r in rs if r['round'] == x])) for x in rounds]
        axes[2].plot(mb, acc, '-o', ms=3, label=m)
    axes[0].set(xlabel='round', ylabel='mean accuracy', title='Accuracy vs rounds')
    axes[1].set(xlabel='round', ylabel='mean macro F1', title='F1 vs rounds')
    axes[2].set(xlabel='cumulative communication (MB)', ylabel='mean accuracy',
                title='Accuracy vs communication')
    axes[2].set_xscale('log')
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig('convergence_communication.png', dpi=140, bbox_inches='tight')
    plt.show()

nd = rows(experiment='new_disease')
priv = rows(experiment='privacy')
abl = [r for r in RESULTS if r.get('algo') == 'feddistill' and r.get('sync_clients') is True
       and r.get('setting') in ('main', 'ablation')]
panels = sum(x != [] for x in [nd, priv, abl])
if panels:
    fig, axes = plt.subplots(1, panels, figsize=(5.5 * panels, 4.2))
    axes = np.atleast_1d(axes)
    k = 0
    if nd:
        for init in sorted(set(r['method'] for r in nd)):
            fr = sorted(set(r['fraction'] for r in nd if r['method'] == init))
            acc = [float(np.mean([r['accuracy'] for r in nd if r['method'] == init and r['fraction'] == f]))
                   for f in fr]
            axes[k].plot(fr, acc, '-o', label=init)
        axes[k].set(xlabel='fraction of new-disease training data', ylabel='accuracy',
                    title=f'Bootstrapping {CONFIG["new_disease"]}')
        axes[k].legend(fontsize=8); axes[k].grid(alpha=0.3); k += 1
    if priv:
        lv = sorted(set(r['dp_noise'] for r in priv))
        acc = [float(np.mean([r['accuracy'] for r in priv if r['dp_noise'] == v])) for v in lv]
        f1 = [float(np.mean([r['f1'] for r in priv if r['dp_noise'] == v])) for v in lv]
        axes[k].plot(lv, acc, '-o', label='accuracy')
        axes[k].plot(lv, f1, '-s', label='macro F1')
        axes[k].set(xlabel='noise multiplier', ylabel='score', title='Noise vs utility')
        axes[k].legend(fontsize=8); axes[k].grid(alpha=0.3); k += 1
    if abl:
        sizes = sorted(set(r['n_public'] for r in abl))
        acc = [float(np.mean([r['accuracy'] for r in abl if r['n_public'] == s])) for s in sizes]
        up = [float(np.mean([r['up_mb'] for r in abl if r['n_public'] == s])) for s in sizes]
        axes[k].plot(sizes, acc, '-o', color='tab:blue')
        axes[k].set(xlabel='reference set size', ylabel='accuracy', title='FedDF reference-set ablation')
        ax2 = axes[k].twinx()
        ax2.plot(sizes, up, '--s', color='tab:red')
        ax2.set_ylabel('uplink MB', color='tab:red')
        axes[k].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('tradeoffs.png', dpi=140, bbox_inches='tight')
    plt.show()
else:
    print('no trade-off panels to plot')

---
## 18. Discussion, limitations and future work

*This section is written after the run. It contains no numbers until the tables above have actually
been produced — fill it in from `results.json` rather than from expectation.*

### Questions the experiments answer

- **Multiclass vs binary.** Compare Table 8 across `classes = 2, 3, 4, native`. The binary framing in
  the original notebook is a simplification, and the centralized/federated gap at native granularity
  is the honest measure of how hard the task really is.
- **Why federate.** Table 2/3 contains all three regimes. The interesting quantity is not
  federated-vs-centralized alone but where federated sits *between* local-only and centralized, per
  client. Small clients (BreastMNIST, RetinaMNIST) are where federation should pay off; large clients
  may lose relative to local-only, which is a real and reportable cost of federation.
- **General-purpose model.** In `union` mode, `outside_rate` says whether the 25-way head can be used
  without knowing the modality of the incoming image. A low within-client accuracy combined with a
  low outside_rate would mean the model has learned modality identity but not pathology.
- **Communication.** Table 4 gives measured MB. FedAvg, FedProx and FedAdam transmit the same volume;
  they differ only in what the server does with it. FedDF differs on the uplink only.
- **New disease.** Table 6 is only meaningful as the three-way comparison. If federated and ImageNet
  arms coincide, the honest conclusion is that federated training over these five domains added
  nothing transferable beyond ImageNet — that is a result, not a failure.

### Limitations to state explicitly

- Five clients, one client per dataset. This conflates "client heterogeneity" with "domain
  heterogeneity"; a realistic hospital federation would have many clients per modality. The
  experiment cannot separate the two effects.
- 28x28 MedMNIST images upsampled to 96x96. Conclusions may not transfer to full-resolution medical
  imaging.
- Training sets capped at `max_train_per_client`, which changes the natural size imbalance between
  clients and therefore the aggregation weights.
- The privacy experiment implements a mechanism, not a guarantee. No epsilon is reported.
- Grad-CAM is a diagnostic, not evidence. No clinical interpretation is offered.
- Runs use `CONFIG['seeds']`; with a single seed the differences between the weight-averaging methods
  are likely within run-to-run variance, and should be described that way rather than ranked.

### Protocol notes on three measurements that are easy to get wrong

- **Grad-CAM agreement.** A CAM whose ReLU leaves every pixel at zero normalizes to a constant map,
  and a quantile threshold then selects the whole image. Two such maps score IoU 1.0 against each
  other and 0.2 against any real map, which looks like a result and is not one. Degenerate maps are
  detected and excluded, the count of excluded maps is reported alongside every IoU, and the
  Grad-CAM target is the true label rather than the model's own prediction.
- **Linear probe.** The probe is fitted on the validation split and scored on the test split. Fitting
  on the training split would let each federated backbone score images it was fine-tuned on while
  the ImageNet backbone sees them for the first time, which measures memorization rather than
  representation quality.
- **Distillation with `sync_clients=False`.** With synchronization off the clients receive nothing
  from the server, so those five models are local-only models trained for the same budget; only the
  server model is a federated artefact. The row is labelled accordingly.

### Future work

- DP-SGD with per-example gradient clipping and an RDP accountant (Opacus) to obtain a reportable
  (epsilon, delta), replacing the heuristic mechanism used here.
- Modality-specific input stems over a shared backbone, which is the natural architecture for genuinely
  multimodal federated learning. The current single-stem ResNet18 forces all five modalities through
  identical early filters, and the local-BatchNorm fix is a partial and cheap substitute for that.
- More clients per modality, to separate statistical heterogeneity from domain heterogeneity.
- Partial client participation and non-IID splits within a modality.

### Final research question

The version supported by this design:

> **In a federation of heterogeneous medical imaging domains, how much predictive performance does
> federated transfer learning give up relative to centralized training, how much does it gain over
> isolated local training, and what does it cost in communication and privacy — and does the
> resulting shared representation transfer to a disease no client contributed?**

Refine the wording once the tables are filled: if the new-disease arm shows no advantage over
ImageNet initialization, the transfer clause should be reported as a negative result rather than
dropped from the question.

---
## 19. Fair sample-exposure protocol: Centralized (S) vs Federated (S/n)

Centralized training sees all S samples every epoch. Federated training with n clients gives each
client only S/n samples. Running both for the same number of "epochs" is not a fair comparison
unless the total number of samples processed is equalized too. If a centralized run uses E epochs,
and a federated run uses e local epochs per round for R rounds, then the rule that keeps both sides
seeing the same total amount of data is:

e times R must equal E

This holds no matter how many clients there are. The cell below checks this rule directly by running
Centralized once at E epochs, and Federated once for every (e, R) pair whose product equals E, so the
fairness claim is verified from actual results rather than assumed.

In [ ]:
def factor_pairs(E):
    pairs = []
    for e in range(1, E + 1):
        if E % e == 0:
            pairs.append((e, E // e))
    return pairs

def fair_protocol_experiment(clients, E=32, seed=0, pairs=None, setting='fair_protocol'):
    pairs = pairs or factor_pairs(E)
    n = len(clients['names'])
    S = sum(clients['sizes'])
    print(f'fair protocol | S={S} samples total | n={n} clients | S/n~={S // n} per client | E={E}')

    cent = run_centralized(clients, seed=seed, epochs=E, setting=setting)

    fed_runs = {}
    for e, R in pairs:
        old_e, old_r = CONFIG['local_epochs'], CONFIG['num_rounds']
        CONFIG['local_epochs'], CONFIG['num_rounds'] = e, R
        print(f'  federated e={e:2d} local epochs x R={R:2d} rounds  (e*R={e * R}, target E={E})')
        fed_runs[(e, R)] = run_federated(clients, algo='fedavg', seed=seed, rounds=R,
                                          setting=setting, verbose=False)
        release(fed_runs[(e, R)])
        for row in RESULTS[-n:]:
            row['local_epochs'], row['fair_rounds'], row['e_times_R'] = e, R, e * R
        CONFIG['local_epochs'], CONFIG['num_rounds'] = old_e, old_r
    return cent, fed_runs

if RUN.get('fair_protocol', True) and not already_done(setting='fair_protocol'):
    FAIR_E = CONFIG['local_epochs'] * CONFIG['num_rounds']
    fair_cent, fair_feds = fair_protocol_experiment(CLIENTS, E=FAIR_E, seed=CONFIG['seeds'][0])
    fair_rows = [r for r in RESULTS if r.get('setting') == 'fair_protocol']
    print()
    print(f"{'method':10s} {'e':>3s} {'R':>3s} {'e*R':>4s}  mean_acc  mean_f1")
    for (e, R), run in fair_feds.items():
        sub = [r for r in fair_rows if r['method'] != 'Centralized' and r.get('local_epochs') == e
               and r.get('fair_rounds') == R]
        macc = mean_over_clients(sub, 'accuracy') if sub else None
        mf1 = mean_over_clients(sub, 'f1') if sub else None
        print(f"{'FedAvg':10s} {e:3d} {R:3d} {e * R:4d}  {macc:.4f}    {mf1:.4f}")
    cent_rows = [r for r in fair_rows if r['method'] == 'Centralized']
    print(f"{'Centralized':10s} {'-':>3s} {'-':>3s} {FAIR_E:4d}  "
          f"{mean_over_clients(cent_rows, 'accuracy'):.4f}    {mean_over_clients(cent_rows, 'f1'):.4f}")
    checkpoint(len(RESULTS), campaign='fair')
    save_results()
elif RUN.get('fair_protocol', True):
    print('matched-exposure protocol already done, skipping')
else:
    print('fair S vs S/n protocol not run')


---
## 20. FedMoE and FedMix: how many experts are useful? (Q4)

Both of these are genuinely different training mechanisms from plain FedAvg, not just FedAvg applied
to a bigger model.

FedMoE here keeps the gating network local and personalized per client, never shared with the
server, and averages the expert sub-networks across clients with top-1 sparse routing. The backbone
is federated exactly as in FedAvg, so the comparison against FedAvg isolates the MoE mechanism
rather than confounding it with a change in how much of the network is shared. A second arm with
`share_backbone=False` shares only the experts, which is the ablation that separates those two
effects. Both run in the same `union` label space as every other method in the notebook, so the
accuracies sit on one scale.

FedMix, following Yoon et al., does not share model weights for its core idea. Each client computes
one averaged image and one averaged label distribution from its own data, shares only that single
summary with the others, and mixes both into local training as a regularizer: the mixed target is
`(1 - lam) * y_own + lam * y_mean_other`, matching the mixed input. In the version below the model
weights are still FedAvg-aggregated round to round for practicality, but the loss function itself
carries the actual FedMix contribution, which is what should be credited as "FedMix" in the
results, not the aggregation step. Mixing label distributions between clients only means something
in a shared output space, so this implementation refuses to run in `native` mode.

In [ ]:
class SparseMoEHead(nn.Module):
    def __init__(self, in_dim=512, hidden=128, num_experts=3, top_k=1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.gate = nn.Linear(in_dim, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.3))
            for _ in range(num_experts)
        ])

    def forward(self, feats, return_gate=False):
        # Switch-Transformer style routing: softmax over ALL experts first, then keep the top-k
        # probability without renormalizing. Taking topk on the raw logits and softmaxing those
        # gives a constant 1.0 when top_k == 1, so no gradient reaches self.gate and the routing
        # never leaves its random initialization.
        gate_logits = self.gate(feats)
        probs = F.softmax(gate_logits, dim=1)
        topk_val, topk_idx = probs.topk(self.top_k, dim=1)
        sparse_w = torch.zeros_like(probs).scatter(1, topk_idx, topk_val)
        expert_out = torch.stack([e(feats) for e in self.experts], dim=1)
        mixed = (sparse_w.unsqueeze(-1) * expert_out).sum(dim=1)
        if return_gate:
            return mixed, sparse_w
        return mixed

class FedMoEModel(nn.Module):
    def __init__(self, num_classes, num_experts=3, top_k=1, pretrained=True, freeze_stem=True):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet18(weights=weights)
        backbone.fc = nn.Identity()
        self.backbone = backbone
        if freeze_stem:
            for p in self.backbone.conv1.parameters():
                p.requires_grad = False
            for p in self.backbone.bn1.parameters():
                p.requires_grad = False
        self.moe = SparseMoEHead(in_dim=512, hidden=128, num_experts=num_experts, top_k=top_k)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x, return_gate=False):
        feats = self.backbone(x)
        mixed, gate_weights = self.moe(feats, return_gate=True)
        out = self.classifier(mixed)
        if return_gate:
            return out, gate_weights
        return out

def fedmoe_federated_names(model, share_backbone=True, share_head=True):
    # The gate stays local (that is the personalization). With share_backbone=True the backbone and
    # the final classifier travel exactly as in FedAvg, so FedMoE differs from FedAvg by the MoE
    # mechanism alone. share_backbone=False is the ablation where only the experts travel.
    bn = bn_param_names(model)
    out = []
    for n, p in model.named_parameters():
        if not p.requires_grad or n in bn or n.startswith('moe.gate'):
            continue
        if n.startswith('moe.experts.'):
            out.append(n)
        elif share_backbone and n.startswith('backbone.'):
            out.append(n)
        elif share_backbone and share_head and n.startswith('classifier.'):
            out.append(n)
    return out

def run_fedmoe(clients, num_experts=3, top_k=1, seed=0, rounds=None, share_backbone=True,
               setting='fedmoe'):
    set_seed(seed)
    rounds = rounds or CONFIG['num_rounds']
    n = len(clients['names'])

    locals_ = [FedMoEModel(head_size(clients, i), num_experts=num_experts, top_k=top_k).to(DEVICE)
               for i in range(n)]
    opts = [optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=CONFIG['learning_rate'])
            for m in locals_]
    global_model = FedMoEModel(head_size(clients, 0), num_experts=num_experts, top_k=top_k).to(DEVICE)
    fed_names = fedmoe_federated_names(global_model, share_backbone=share_backbone,
                                       share_head=(clients['label_space'] != 'native'))
    comm = Comm()
    label = f'FedMoE (K={num_experts},top{top_k},bb={int(share_backbone)})'

    t0 = time.time()
    metrics = None
    for rnd in range(1, rounds + 1):
        g_state = extract_state(global_model, fed_names)
        nbytes = state_bytes(g_state)
        for m in locals_:
            load_state(m, g_state)
            comm.receive(nbytes)

        for i, m in enumerate(locals_):
            train_local(m, clients['train_loaders'][i], opts[i], CONFIG['local_epochs'])

        client_states = [extract_state(m, fed_names) for m in locals_]
        for s in client_states:
            comm.send(state_bytes(s))
        load_state(global_model, aggregate(client_states, clients['sizes']))

        metrics = [evaluate_client(m, ld, ids)
                   for m, ld, ids in zip(locals_, clients['test_loaders'], clients['class_ids'])]
        for name, mt in zip(clients['names'], metrics):
            record(HISTORY, method=label, algo='fedmoe', seed=seed, round=rnd, client=name,
                   label_space=clients['label_space'], accuracy=mt['accuracy'], f1=mt['f1'],
                   cum_mb=comm.total_mb(), elapsed_s=time.time() - t0)
        print(f'  {label} r{rnd:02d} | mean acc {mean_over_clients(metrics, "accuracy"):.4f} | '
              f'mean f1 {mean_over_clients(metrics, "f1"):.4f}')

    elapsed = time.time() - t0
    summary = comm.summary(rounds, n)

    # Q4 asks how many experts are useful, so the routing itself has to be measured and not just
    # the accuracy. Three quantities are recorded: the mean gate vector per client, the share of
    # test images each expert actually receives across the whole federation, and the entropy of
    # each client's gate. An expert that receives less than 1 % of the traffic is counted as idle:
    # with five clients and top-1 routing at most five experts can ever be occupied, so a large K
    # is expected to leave experts idle and the measurement shows whether that is what happens.
    gate_table = []
    usage_counts = torch.zeros(num_experts)
    total_images = 0
    for i, (name, m) in enumerate(zip(clients['names'], locals_)):
        m.eval()
        weights_sum, count = torch.zeros(num_experts), 0
        client_counts = torch.zeros(num_experts)
        with torch.no_grad():
            for imgs, _ in clients['test_loaders'][i]:
                _, gw = m(imgs.to(DEVICE), return_gate=True)
                weights_sum += gw.sum(dim=0).cpu()
                picks = gw.argmax(dim=1).cpu()
                client_counts += torch.bincount(picks, minlength=num_experts).float()
                count += gw.shape[0]
        mean_gate = (weights_sum / max(count, 1)).tolist()
        usage_counts += client_counts
        total_images += count
        dominant = int(np.argmax(mean_gate))
        p = np.array(mean_gate, dtype=float)
        p = p / max(p.sum(), 1e-12)
        entropy = float(-(p * np.log(p + 1e-12)).sum())
        client_share = (client_counts / max(count, 1)).tolist()
        gate_table.append({'client': name, 'modality': MODALITY[name], 'dominant_expert': dominant,
                           'gate_entropy': round(entropy, 4),
                           **{f'expert_{k}': round(mean_gate[k], 4) for k in range(num_experts)}})
        record(RESULTS, experiment='fedmoe', method=label, client=name, seed=seed,
               label_space=clients['label_space'], classes=clients['num_classes'][i],
               setting=setting, num_experts=num_experts, top_k=top_k,
               share_backbone=share_backbone, dominant_expert=dominant, gate_weights=mean_gate,
               gate_entropy=entropy, client_expert_share=[round(v, 4) for v in client_share],
               time_s=elapsed, up_mb=summary['up_mb'], down_mb=summary['down_mb'], rounds=rounds,
               **{k: v for k, v in metrics[i].items()
                  if k not in ('confusion', 'per_class_recall', 'support')})

    usage = (usage_counts / max(total_images, 1)).tolist()
    experts_used = int(sum(1 for u in usage if u >= 0.01))
    distinct_dominant = len({row['dominant_expert'] for row in gate_table})
    mean_entropy = float(np.mean([row['gate_entropy'] for row in gate_table]))
    for r in RESULTS:
        if r.get('experiment') == 'fedmoe' and r.get('setting') == setting and r.get('seed') == seed:
            r['experts_used'] = experts_used
            r['distinct_dominant'] = distinct_dominant
            r['expert_usage'] = [round(u, 4) for u in usage]
            r['mean_gate_entropy'] = mean_entropy

    print(f'{label} | mean acc {mean_over_clients(metrics, "accuracy"):.4f} | '
          f'mean f1 {mean_over_clients(metrics, "f1"):.4f} | '
          f'{summary["total_mb"]:.1f} MB | {elapsed:.0f}s')
    print(f'  experts receiving >=1% of images: {experts_used} of {num_experts} | '
          f'distinct dominant experts: {distinct_dominant} | mean gate entropy {mean_entropy:.3f}')
    print(f'  expert usage share over all clients: {[round(u, 3) for u in usage]}')
    print('  per-client gate weights (local, personalized, never shared with the server):')
    for row in gate_table:
        print('   ', row)
    return {'locals': locals_, 'global': global_model, 'metrics': metrics, 'gate_table': gate_table,
            'comm': summary, 'time_s': elapsed, 'label': label, 'experts_used': experts_used,
            'usage': usage, 'mean_gate_entropy': mean_entropy}


class MeanBatchStore:
    def __init__(self, clients, batches_per_client=5, seed=0):
        set_seed(seed)
        self.num_classes = clients['total_classes']
        self.mean_images, self.mean_labels = {}, {}
        for i, name in enumerate(clients['names']):
            loader = DataLoader(clients['train'][i], batch_size=CONFIG['batch_size'], shuffle=True)
            imgs_acc, labels_acc, n = None, None, 0
            for b, (imgs, labels) in enumerate(loader):
                if b >= batches_per_client:
                    break
                oh = F.one_hot(labels, num_classes=self.num_classes).float()
                bi, bl = imgs.mean(dim=0, keepdim=True), oh.mean(dim=0, keepdim=True)
                imgs_acc = bi if imgs_acc is None else imgs_acc + bi
                labels_acc = bl if labels_acc is None else labels_acc + bl
                n += 1
            self.mean_images[name] = (imgs_acc / max(n, 1)).squeeze(0)
            self.mean_labels[name] = (labels_acc / max(n, 1)).squeeze(0)

    def sample_other(self, exclude_name, rng):
        others = [n for n in self.mean_images if n != exclude_name]
        pick = others[rng.randrange(len(others))]
        return self.mean_images[pick], self.mean_labels[pick]

def fedmix_loss(model, imgs, labels, mean_store, client_name, num_classes, lam=0.2, rng=None):
    rng = rng or random
    ce = F.cross_entropy(model(imgs), labels)
    mean_img, mean_label = mean_store.sample_other(client_name, rng)
    mixed_imgs = (1 - lam) * imgs + lam * mean_img.unsqueeze(0).to(imgs.device)
    log_probs = F.log_softmax(model(mixed_imgs), dim=1)
    true_oh = F.one_hot(labels, num_classes=num_classes).float()
    # both halves of the FedMix target: the client's own label and the shared mean label of the
    # other client whose average image was mixed in.
    target = (1 - lam) * true_oh + lam * mean_label.unsqueeze(0).to(imgs.device)
    return ce + lam * (-(target * log_probs).sum(dim=1).mean())

def train_local_fedmix(model, loader, opt, epochs, mean_store, client_name, num_classes,
                       lam=0.2, seed=0):
    rng = random.Random(seed)
    model.train()
    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = fedmix_loss(model, imgs, labels, mean_store, client_name, num_classes,
                               lam=lam, rng=rng)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

def run_fedmix(clients, lam=0.2, seed=0, rounds=None, setting='fedmix'):
    if clients['label_space'] == 'native':
        raise ValueError('FedMix mixes label distributions between clients, which needs a shared '
                         'output space; run it in union or binary mode')
    set_seed(seed)
    rounds = rounds or CONFIG['num_rounds']
    n = len(clients['names'])
    num_classes = clients['total_classes']
    mean_store = MeanBatchStore(clients, seed=seed)

    locals_ = [FederatedModel(head_size(clients, i)).to(DEVICE) for i in range(n)]
    opts = [optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=CONFIG['learning_rate'])
            for m in locals_]
    global_model = FederatedModel(head_size(clients, 0)).to(DEVICE)
    fed_names = federated_names(global_model)
    comm = Comm()
    label = f'FedMix (lam={lam})'

    t0 = time.time()
    metrics = None
    for rnd in range(1, rounds + 1):
        g_state = extract_state(global_model, fed_names)
        nbytes = state_bytes(g_state)
        for m in locals_:
            load_state(m, g_state)
            comm.receive(nbytes)

        for i, m in enumerate(locals_):
            train_local_fedmix(m, clients['train_loaders'][i], opts[i], CONFIG['local_epochs'],
                               mean_store, clients['names'][i], num_classes, lam=lam, seed=seed + rnd)

        client_states = [extract_state(m, fed_names) for m in locals_]
        for s in client_states:
            comm.send(state_bytes(s))
        load_state(global_model, aggregate(client_states, clients['sizes']))

        metrics = [evaluate_client(m, ld, ids)
                   for m, ld, ids in zip(locals_, clients['test_loaders'], clients['class_ids'])]
        for name, mt in zip(clients['names'], metrics):
            record(HISTORY, method=label, algo='fedmix', seed=seed, round=rnd, client=name,
                   label_space=clients['label_space'], accuracy=mt['accuracy'], f1=mt['f1'],
                   cum_mb=comm.total_mb(), elapsed_s=time.time() - t0)
        print(f'  {label} r{rnd:02d} | mean acc {mean_over_clients(metrics, "accuracy"):.4f} | '
              f'mean f1 {mean_over_clients(metrics, "f1"):.4f}')

    elapsed = time.time() - t0
    summary = comm.summary(rounds, n)
    for i, (name, mt) in enumerate(zip(clients['names'], metrics)):
        record(RESULTS, experiment='fedmix', method=label, client=name, seed=seed,
               label_space=clients['label_space'], classes=clients['num_classes'][i],
               setting=setting, lam=lam, time_s=elapsed,
               up_mb=summary['up_mb'], down_mb=summary['down_mb'], rounds=rounds,
               **{k: v for k, v in mt.items() if k not in ('confusion', 'per_class_recall', 'support')})
    print(f'{label} | mean acc {mean_over_clients(metrics, "accuracy"):.4f} | '
          f'{summary["total_mb"]:.1f} MB | {elapsed:.0f}s')
    return {'locals': locals_, 'global': global_model, 'metrics': metrics, 'comm': summary,
            'time_s': elapsed, 'label': label}

def _checkpoint(start, campaign='union', run=None):
    """Label and save the rows written since `start`, then release the run's weights. Called
    after every individual run so that an interrupted sweep never loses what already finished and
    a long sweep does not accumulate a dozen models on the GPU."""
    tag_rows(start, campaign=campaign)
    save_results()
    if run is not None:
        release(run)
        gpu_report('after release:')
    return len(RESULTS)

if RUN.get('moe_mix', True):
    _start = len(RESULTS)
    FEDMOE_RUNS = {}
    _have = {r['setting'] for r in RESULTS if r.get('experiment') == 'fedmoe'}

    # Top-1 routing across the whole expert range. Five clients cannot occupy more than five
    # experts under top-1, so K = 6, 7 and 10 test whether the extra capacity is used or idle.
    for K in CONFIG['expert_counts']:
        if f'fedmoe_K{K}' in _have:
            print(f'fedmoe_K{K} already present, skipping')
            continue
        FEDMOE_RUNS[f'K{K}'] = run_fedmoe(CLIENTS, num_experts=K, top_k=1,
                                          seed=CONFIG['seeds'][0], share_backbone=True,
                                          setting=f'fedmoe_K{K}')
        _start = _checkpoint(_start, run=FEDMOE_RUNS[f'K{K}'])

    # Top-2 routing at the two largest expert counts. If the ceiling on useful experts is imposed
    # by top-1 routing rather than by the data, allowing each client to mix two experts should
    # raise the number of experts that receive traffic.
    for K in CONFIG['expert_counts_top2']:
        if f'fedmoe_K{K}_top2' in _have:
            print(f'fedmoe_K{K}_top2 already present, skipping')
            continue
        FEDMOE_RUNS[f'K{K}_top2'] = run_fedmoe(CLIENTS, num_experts=K, top_k=2,
                                               seed=CONFIG['seeds'][0], share_backbone=True,
                                               setting=f'fedmoe_K{K}_top2')
        _start = _checkpoint(_start, run=FEDMOE_RUNS[f'K{K}_top2'])

    # Ablation: share only the experts, never the backbone. Isolates the MoE mechanism from the
    # amount of the network that travels.
    if 'fedmoe_K3_nobb' not in _have:
        FEDMOE_RUNS['K3_nobb'] = run_fedmoe(CLIENTS, num_experts=3, top_k=1,
                                            seed=CONFIG['seeds'][0], share_backbone=False,
                                            setting='fedmoe_K3_nobb')
        _start = _checkpoint(_start, run=FEDMOE_RUNS['K3_nobb'])

    if any(r.get('experiment') == 'fedmix' and r.get('setting') == 'fedmix' for r in RESULTS):
        print('fedmix already present, skipping')
        FEDMIX_RUN = None
    else:
        FEDMIX_RUN = run_fedmix(CLIENTS, lam=0.2, seed=CONFIG['seeds'][0], setting='fedmix')
        _start = _checkpoint(_start, run=FEDMIX_RUN)

    print()
    print('Q4 summary, superset label space')
    print(f"{'method':34s} {'acc':>7s} {'F1':>7s} {'AUC':>7s} {'used':>6s} {'ent':>6s} {'MB':>9s}")
    fedavg_rows = [r for r in RESULTS if r.get('method') == 'FedAvg' and r.get('setting') == 'main']
    if fedavg_rows:
        print(f"{'FedAvg (baseline)':34s} {mean_over_clients(fedavg_rows, 'accuracy'):7.4f} "
              f"{mean_over_clients(fedavg_rows, 'f1'):7.4f} "
              f"{_fmt(mean_over_clients(fedavg_rows, 'auc'))} {'-':>6s} {'-':>6s} {'-':>9s}")
    for key, run in FEDMOE_RUNS.items():
        print(f"{run['label']:34s} {mean_over_clients(run['metrics'], 'accuracy'):7.4f} "
              f"{mean_over_clients(run['metrics'], 'f1'):7.4f} "
              f"{_fmt(mean_over_clients(run['metrics'], 'auc'))} "
              f"{run['experts_used']:>3d}/{len(run['usage']):<2d} "
              f"{run['mean_gate_entropy']:6.3f} {run['comm']['total_mb']:9.1f}")
    if FEDMIX_RUN is not None:
        print(f"{FEDMIX_RUN['label']:34s} "
              f"{mean_over_clients(FEDMIX_RUN['metrics'], 'accuracy'):7.4f} "
              f"{mean_over_clients(FEDMIX_RUN['metrics'], 'f1'):7.4f} "
              f"{_fmt(mean_over_clients(FEDMIX_RUN['metrics'], 'auc'))} {'-':>6s} {'-':>6s} "
              f"{FEDMIX_RUN['comm']['total_mb']:9.1f}")
else:
    print('FedMoE / FedMix comparison not run')


---
## 21. Bootstrap model as a repeatable method: Leave-One-Client-Out (LOCO)

Instead of relying on one external test, the bootstrap idea is turned into something repeatable
across all five clients. For each client held out in turn: pretrain federatively on the other four,
fine-tune the result on the held-out client's own data at several fractions of that data (including
zero, which is the zero-shot case examined in the next section), then evaluate. This gives five real
data points instead of one.

Each fine-tuning fraction is run twice, from the federated backbone and from plain ImageNet weights.
Without that control an apparent gain from the federated initialization could simply be the ImageNet
pretraining the federated model itself started from -- the same control used in Section 14.

The head spans the shared 25-class output space and the metric restricts the argmax to the held-out
client's own block of class ids. A head sized to the client's native class count cannot be scored
against union-offset labels at all, which is worth stating because it is exactly the kind of
mismatch that produces a meaningless zero.

In [ ]:
LOCO_SOURCES = {}

def transplant_backbone(model, source_model):
    src, dst = source_model.state_dict(), model.state_dict()
    for k in dst:
        if k.startswith('backbone.') and 'bn' not in k and 'downsample.1' not in k and k in src:
            dst[k] = src[k].clone()
    model.load_state_dict(dst)
    return model

def majority_baseline(test_ds, class_ids):
    """Accuracy of the constant predictor that always answers the most frequent class in the test
    set. A zero-shot model has demonstrated nothing until it beats this floor."""
    ys = [y.numpy() for _, y in DataLoader(test_ds, batch_size=256, shuffle=False)]
    y = np.concatenate(ys)
    counts = np.bincount(y, minlength=int(max(class_ids)) + 1)[np.array(class_ids)]
    return float(counts.max() / max(counts.sum(), 1))

def zero_shot_model(source_model, head_n, arm, seed=0):
    """Build a model for the zero-shot (0 % local data) point of the bootstrap experiment.

    imagenet            nothing is transplanted; the ImageNet control.
    federated-backbone  convolutional weights only, classification head left random. This is the
                        only arm that is meaningful in the superset label space, because there the
                        held-out client's block of the output layer never received a gradient.
    federated-full      convolutional weights and the classification head. Only meaningful when
                        every client shares one output space, i.e. the binary setting, where the
                        head learned by the other four clients applies directly to the fifth.

    BatchNorm is left at its ImageNet values in every arm, because BatchNorm is never federated in
    this study and the held-out client has no statistics of its own before it sees any data.
    """
    set_seed(seed)
    model = FederatedModel(head_n, pretrained=True).to(DEVICE)
    if arm == 'imagenet':
        return model
    transplant_backbone(model, source_model)
    if arm == 'federated-full':
        src, dst = source_model.state_dict(), model.state_dict()
        for k in dst:
            if k.startswith('head.') and k in src and src[k].shape == dst[k].shape:
                dst[k] = src[k].clone()
        model.load_state_dict(dst)
    return model

def zero_shot_transplant(source_model, test_ds, head_n, client_name, class_ids, seed=0,
                         arm='federated-backbone'):
    model = zero_shot_model(source_model, head_n, arm, seed=seed)
    m = evaluate_client(model, DataLoader(test_ds, batch_size=128, shuffle=False), class_ids)
    m['arm'] = arm
    print(f'  0%  {arm:20s} {client_name:16s} head={head_n} acc {m["accuracy"]:.4f} '
          f'f1 {m["f1"]:.4f}')
    return m

def finetune_from_source(source_model, train_ds, test_ds, head_n, class_ids, fraction,
                         seed=0, init='federated', epochs=None):
    set_seed(seed)
    epochs = epochs or CONFIG['new_disease_epochs']
    idx = np.random.RandomState(seed).choice(len(train_ds),
                                             max(int(len(train_ds) * fraction), 16), replace=False)
    loader = DataLoader(Subset(train_ds, idx.tolist()), batch_size=CONFIG['batch_size'], shuffle=True)
    model = FederatedModel(head_n, pretrained=True).to(DEVICE)
    if init == 'federated':
        transplant_backbone(model, source_model)
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG['learning_rate'])
    train_local(model, loader, opt, epochs)
    m = evaluate_client(model, DataLoader(test_ds, batch_size=128, shuffle=False), class_ids)
    m['n_train'] = len(idx)
    return m

def loco_pretrain(clients, held_out_idx, seed=0, algo='fedadam'):
    keep = [i for i in range(len(clients['names'])) if i != held_out_idx]
    sub = {key: ([clients[key][i] for i in keep] if isinstance(clients[key], list) else clients[key])
           for key in clients}
    run = run_federated(sub, algo=algo, seed=seed, rounds=CONFIG['num_rounds'],
                        setting=f'loco_pretrain_holdout_{clients["names"][held_out_idx]}',
                        verbose=False)
    return run['global']

def loco_bootstrap(clients, fractions=(0.0, 0.1, 0.25, 0.5, 1.0), seed=0, algo='fedadam',
                   setting='bootstrap_loco'):
    table = []
    for held_out_idx, held_name in enumerate(clients['names']):
        # A finished holdout writes one row per fraction per arm. Skipping it on a re-run is what
        # makes an interrupted bootstrap cheap to resume: each holdout is a fresh federated
        # pretraining of four clients plus several fine-tunings.
        if already_done(experiment='bootstrap_loco', setting=setting, client=held_name,
                        fraction=max(fractions), init='federated'):
            print(f'=== LOCO holdout: {held_name} already done, skipping ===')
            continue
        print(f'=== LOCO holdout: {held_name} ({clients["label_space"]} label space) ===')
        source_model = loco_pretrain(clients, held_out_idx, seed=seed, algo=algo)
        # Cached so the seed ablation in the next section does not have to pretrain again.
        LOCO_SOURCES[(clients['label_space'], held_name)] = source_model
        train_ds, test_ds = clients['train'][held_out_idx], clients['test'][held_out_idx]
        # the head must span the shared output space, and the metric must use the client's own
        # block of class ids inside it -- a per-client-sized head cannot be scored against
        # union-offset labels.
        head_n = head_size(clients, held_out_idx)
        class_ids = clients['class_ids'][held_out_idx]
        # The floor for the 0 % point, so that a zero-shot number can be read against something.
        base = majority_baseline(test_ds, class_ids)
        print(f'  majority-class baseline for {held_name}: {base:.4f}')
        record(RESULTS, experiment='bootstrap_loco', method='Majority baseline', client=held_name,
               seed=seed, fraction=0.0, classes=head_n, setting=setting,
               label_space=clients['label_space'], init='majority', accuracy=base)

        for frac in fractions:
            # Q6: at 0 % the three arms separate the contribution of the backbone from the
            # contribution of the head. At every other fraction the ImageNet arm is the control
            # that stops a gain being credited to federation when it came from ImageNet.
            arms = (['imagenet', 'federated-backbone', 'federated-full'] if frac == 0.0
                    else ['imagenet', 'federated'])
            for init in arms:
                if frac == 0.0:
                    m = zero_shot_transplant(source_model, test_ds, head_n, held_name,
                                             class_ids=class_ids, seed=seed, arm=init)
                else:
                    m = finetune_from_source(source_model, train_ds, test_ds, head_n, class_ids,
                                             frac, seed=seed, init=init)
                    print(f'  {init:10s} frac {frac:<5} n={m["n_train"]:<5} '
                          f'acc {m["accuracy"]:.4f} f1 {m["f1"]:.4f}')
                table.append({'client': held_name, 'fraction': frac, 'init': init, **m})
                record(RESULTS, experiment='bootstrap_loco', method=f'Bootstrapped-{init}',
                       client=held_name, seed=seed, fraction=frac, classes=head_n,
                       setting=setting, label_space=clients['label_space'], init=init,
                       majority_baseline=base,
                       **{k: v for k, v in m.items()
                          if k not in ('confusion', 'per_class_recall', 'support', 'arm')})
        # Checkpoint so an interruption costs at most the client that was in flight, and drop
        # that client's pretrained model, which is not needed once its rows are recorded.
        save_results()
        LOCO_SOURCES.pop((clients['label_space'], held_name), None)
        release({'model': source_model})
    return table

if RUN.get('bootstrap', True):
    _start = len(RESULTS)
    LOCO_TABLE = loco_bootstrap(CLIENTS, fractions=CONFIG['bootstrap_fractions'],
                                seed=CONFIG['seeds'][0], algo=CONFIG['bootstrap_algo'])
    for r in RESULTS[_start:]:
        r['campaign'] = 'union'
    print()
    print(f"{'client':16s} {'frac':>5s} {'init':20s}  acc      f1")
    if not LOCO_TABLE:
        print('  (all holdouts were already recorded; see the saved results)')
    for row in LOCO_TABLE:
        print(f"{row['client']:16s} {row['fraction']:>5} {row['init']:20s}  "
              f"{row['accuracy']:.4f}  {row['f1']:.4f}")
    save_results()
else:
    print('LOCO bootstrap campaign not run')


---
## 22. What does a bootstrap model achieve with zero local data? (Q6)

When a backbone pretrained on four clients is transplanted to a fifth, held-out client with no
fine-tuning steps at all, its classification head is left at its fresh random initialization -- it
has never seen that client's label mapping. A random linear layer over a handful of classes does not
sit at chance accuracy in general; depending on the random draw, its output can concentrate on one or
two classes, and if none of those happen to be correct for any test example, measured accuracy is
exactly 0 percent. That is an expected property of zero-shot head transplantation, not automatically
a bug -- but it should be checked, not assumed, and it is not the only explanation available. A head
whose output space does not match the label space of the test set also scores exactly 0 percent, and
that failure looks identical from the outside.

The cell below therefore repeats the zero-shot evaluation over several seeds for every held-out
client and prints, for each run, how the predictions are distributed over the client's classes next
to the true distribution. A collapse onto one or two classes is evidence for the random-head
explanation; predictions spread across all classes at 0 percent accuracy would rule it out and point
somewhere else.

In [ ]:
def zero_shot_probe(clients, held_out_idx, seeds=(0, 1, 2), algo='fedadam'):
    held_name = clients['names'][held_out_idx]
    cached = LOCO_SOURCES.get((clients['label_space'], held_name))
    source_model = cached if cached is not None else loco_pretrain(clients, held_out_idx,
                                                                   seed=0, algo=algo)
    test_ds = clients['test'][held_out_idx]
    head_n = head_size(clients, held_out_idx)
    ids = clients['class_ids'][held_out_idx]
    loader = DataLoader(test_ds, batch_size=128, shuffle=False)
    accs = []
    for s in seeds:
        set_seed(s)
        model = transplant_backbone(FederatedModel(head_n, pretrained=True).to(DEVICE), source_model)
        y, prob = collect_predictions(model, loader)
        pred = np.array(ids)[prob[:, ids].argmax(1)]
        acc = float((pred == y).mean())
        accs.append(acc)
        spread = np.bincount(pred, minlength=head_n)[ids]
        truth = np.bincount(y, minlength=head_n)[ids]
        print(f'  {held_name:16s} seed={s} acc {acc:.4f} | predicted {spread} | true {truth}')
        record(RESULTS, experiment='zero_shot_ablation', method='Bootstrapped-zero-shot',
               client=held_name, seed=s, fraction=0.0, classes=head_n,
               setting='zero_shot_ablation', label_space=clients['label_space'],
               accuracy=acc, pred_spread=spread.tolist())
    a = np.array(accs)
    print(f'  -> {held_name}: mean={a.mean():.4f} std={a.std():.4f} all_zero={bool((a == 0).all())}')
    return a

if RUN.get('zero_shot_ablation', True):
    ZS = {}
    for i, nm in enumerate(CLIENTS['names']):
        ZS[nm] = zero_shot_probe(CLIENTS, i, seeds=(0, 1, 2))
    print()
    print('Reading the result: if the predicted spread concentrates on one or two of the client\'s '
          'classes across every seed, the 0 percent is the untrained random head collapsing onto a '
          'class that is absent or rare in the test set -- an expected property of zero-shot head '
          'transplantation. If accuracy is 0 while predictions are spread over all of the client\'s '
          'classes, the cause is not the random head and must be investigated separately.')
else:
    print('zero-shot seed ablation not run')


---
## 23. Scalability: how methods behave as the number of clients changes

None of the experiments above vary how many clients participate -- they always use all five. This
section subsamples smaller groups of clients (2, 3, 4, and the full 5) from the existing five and
reruns FedAvg on each subset, so accuracy, communication cost, and per-round time can be compared as
n grows. This is a direct, empirical answer to "does this approach scale," rather than an assumption
based on the five-client case alone.

In [ ]:
def scalability_sweep(clients, client_counts=(2, 3, 4, 5), seed=0, algo='fedavg', setting='scalability'):
    all_names = clients['names']
    results_rows = []
    for k in client_counts:
        keep = list(range(min(k, len(all_names))))
        sub = {key: [clients[key][i] for i in keep] if isinstance(clients[key], list) else clients[key]
               for key in clients}
        print(f'=== scalability: n={len(keep)} clients ({[all_names[i] for i in keep]}) ===')
        run = run_federated(sub, algo=algo, seed=seed, rounds=CONFIG['num_rounds'],
                             setting=f'{setting}_n{len(keep)}', verbose=False)
        macc = mean_over_clients(run['metrics'], 'accuracy')
        mf1 = mean_over_clients(run['metrics'], 'f1')
        row = {'n_clients': len(keep), 'mean_acc': macc, 'mean_f1': mf1,
               'total_mb': run['comm']['total_mb'], 'time_s': run['time_s']}
        release(run)
        save_results()
        results_rows.append(row)
        print(f'  n={len(keep)} | acc {macc:.4f} | f1 {mf1:.4f} | {run["comm"]["total_mb"]:.1f} MB | '
              f'{run["time_s"]:.1f}s')
    return results_rows

if RUN.get('scalability', True) and not already_done(setting='scalability_n5'):
    SCALABILITY_TABLE = scalability_sweep(CLIENTS, client_counts=(2, 3, 4, 5), seed=CONFIG['seeds'][0])
    checkpoint(len(RESULTS), campaign='scalability')
    save_results()
elif RUN.get('scalability', True):
    print('scalability sweep already done, skipping')
    SCALABILITY_TABLE = []
    print()
    print(f"{'n_clients':>9s} {'mean_acc':>9s} {'mean_f1':>8s} {'total_MB':>9s} {'time_s':>7s}")
    for row in SCALABILITY_TABLE:
        print(f"{row['n_clients']:9d} {row['mean_acc']:9.4f} {row['mean_f1']:8.4f} "
              f"{row['total_mb']:9.1f} {row['time_s']:7.1f}")
    print()
    print('If accuracy stays roughly flat and communication/time grow close to linearly with n, '
          'the method scales cleanly. If accuracy drops noticeably as n grows, that points to '
          'non-IID aggregation noise increasing with more, smaller client splits.')
else:
    print('scalability sweep not run')


---
## 24. Reliability: how stable are results across repeated runs

A single run with a single random seed can look better or worse than the method actually is, purely
by chance in weight initialization, batch ordering, and client sampling. This section repeats the
main FedAvg run across several seeds and reports the mean and standard deviation of accuracy per
client, which is a direct, standard way to report how reliable a reported number actually is.

In [ ]:
def reliability_sweep(clients, seeds=(0, 1, 2), algo='fedavg', setting='reliability'):
    per_seed = []
    for s in seeds:
        print(f'=== reliability: seed={s} ===')
        run = run_federated(clients, algo=algo, seed=s, rounds=CONFIG['num_rounds'],
                             setting=f'{setting}_seed{s}', verbose=False)
        per_seed.append(run['metrics'])
        release(run)
        save_results()
    n = len(clients['names'])
    rows = []
    for i, name in enumerate(clients['names']):
        accs = np.array([per_seed[s_idx][i]['accuracy'] for s_idx in range(len(seeds))])
        f1s = np.array([per_seed[s_idx][i]['f1'] for s_idx in range(len(seeds))])
        rows.append({'client': name, 'mean_acc': accs.mean(), 'std_acc': accs.std(),
                     'mean_f1': f1s.mean(), 'std_f1': f1s.std()})
    return rows

if RUN.get('reliability', True) and not already_done(setting='reliability_seed2'):
    RELIABILITY_TABLE = reliability_sweep(CLIENTS, seeds=(0, 1, 2))
    checkpoint(len(RESULTS), campaign='reliability')
    save_results()
elif RUN.get('reliability', True):
    print('reliability sweep already done, skipping')
    RELIABILITY_TABLE = []
    print()
    print(f"{'client':16s} {'mean_acc':>9s} {'std_acc':>8s} {'mean_f1':>8s} {'std_f1':>7s}")
    for row in RELIABILITY_TABLE:
        print(f"{row['client']:16s} {row['mean_acc']:9.4f} {row['std_acc']:8.4f} "
              f"{row['mean_f1']:8.4f} {row['std_f1']:7.4f}")
    print()
    print('A small standard deviation across seeds means the reported accuracy for that client is '
          'stable and trustworthy. A large one means a single-seed number should not be quoted as '
          'the result on its own -- report the mean and standard deviation together instead.')
else:
    print('reliability sweep not run')


---
## 25. Explainability addendum: reading the FedMoE gate as an explanation tool

Grad-CAM and CKA, used earlier in this notebook, explain what a single client's model is looking at
and how similar two clients' representations are. The FedMoE gate weights recorded in Section 20 add
a third, complementary kind of explanation: which clients the model itself has learned to treat as
similar enough to share an expert with, expressed as a probability distribution over experts rather
than a single similarity number. The table below turns the raw gate weights already recorded into a
readable client-to-expert grouping, for direct comparison against the CKA number from earlier in the
notebook.

In [ ]:
def explain_moe_grouping(num_experts=3, share_backbone=True):
    rows_k = [r for r in RESULTS if r.get('experiment') == 'fedmoe'
              and r.get('num_experts') == num_experts
              and r.get('share_backbone') == share_backbone]
    if not rows_k:
        print(f'FedMoE (K={num_experts}, bb={int(share_backbone)}): no records')
        return {}
    groups = {}
    for r in rows_k:
        groups.setdefault(r['dominant_expert'], []).append(r['client'])
    print(f'FedMoE (K={num_experts}, shared backbone={share_backbone}) client-to-expert grouping:')
    for expert_id, client_list in sorted(groups.items()):
        print(f'  expert {expert_id}: {client_list}')
    return groups

if RUN.get('moe_mix', True):
    for K in [2, 3, 4]:
        explain_moe_grouping(num_experts=K, share_backbone=True)
        print()
    explain_moe_grouping(num_experts=3, share_backbone=False)
    print()
    print('Compare this grouping to the CKA matrix from Section 16: if clients that CKA found '
          'similar also share a dominant expert here, the two explanations agree; if not, that '
          'disagreement is itself worth reporting as a finding, not something to reconcile by hand.')


---
## 26. Experiment parameters and workflow, generated from what actually ran

The table and snapshot below are built directly from CONFIG and RESULTS, not typed by hand, so they
describe exactly what ran in this notebook rather than what was planned.

In [ ]:
def experiment_parameter_table():
    S = sum(CLIENTS['sizes'])
    n = len(CLIENTS['names'])
    rows = []
    seen = set()
    for r in RESULTS:
        key = (r.get('experiment'), r.get('setting'), r.get('method'))
        if key in seen:
            continue
        seen.add(key)
        is_centralized = r.get('method') == 'Centralized'
        rows.append({
            'experiment': r.get('experiment'),
            'setting': r.get('setting'),
            'method': r.get('method'),
            'rounds (R)': r.get('rounds', r.get('fair_rounds', '-')),
            'local_epochs (e)': r.get('local_epochs', CONFIG['local_epochs'] if not is_centralized else '-'),
            'samples/step': (S if is_centralized
                             else f'{min(CLIENTS["sizes"])}-{max(CLIENTS["sizes"])}'),
            'num_experts': r.get('num_experts', '-'),
            'fraction': r.get('fraction', '-'),
        })
    return rows

PARAM_TABLE = experiment_parameter_table()
print(f"{'experiment':18s} {'setting':22s} {'method':28s} {'R':>3s} {'e':>3s} {'samples/step':>12s} "
      f"{'K':>3s} {'frac':>5s}")
for row in PARAM_TABLE:
    print(f"{str(row['experiment']):18s} {str(row['setting']):22s} {str(row['method']):28s} "
          f"{str(row['rounds (R)']):>3s} {str(row['local_epochs (e)']):>3s} "
          f"{str(row['samples/step']):>12s} {str(row['num_experts']):>3s} {str(row['fraction']):>5s}")

S = sum(CLIENTS['sizes'])
n = len(CLIENTS['names'])
E = CONFIG['local_epochs'] * CONFIG['num_rounds']
print()
print(f'n_clients: {n}')
print(f'S_total_train_samples: {S}')
print(f'S_over_n_nominal: {round(S / n)}  (nominal only - actual sizes below are unequal)')
print(f'E_centralized_equiv_epochs: {E}')
print('per_client_sizes:', dict(zip(CLIENTS['names'], CLIENTS['sizes'])))


### Workflow, in order

Five clients are built, one per MedMNIST dataset, each with its own train, validation and test
split. The training sets are capped at `max_train_per_client`, so the per-client sizes are
2000 / 546 / 2000 / 2000 / 1080 and S is their sum. They are **not** equal: S/n is the nominal
per-client budget used in the protocol description, while the actual sizes are printed in
Section 26 and used as the aggregation weights. Where a client is smaller than S/n, it contributes
proportionally less to the average, and that is stated rather than hidden.

From that shared client setup the following are run and compared, all in the same `union` label
space so their numbers are on one scale: Centralized (all S samples pooled), Local-only (each
client alone, no communication), FedAvg / FedProx / FedAdam, Federated Distillation, FedMoE and
FedMix (Section 20), and the LOCO bootstrap protocol (Section 21).

Sample exposure is matched by the rule e x R = E from Section 19: the main run uses e = 4 local
epochs over R = 8 rounds, and Centralized and Local-only both use E = 32 epochs, so every arm
performs the same number of passes over the data available to it. Two exceptions are deliberate
and noted here: the LOCO fine-tuning stage uses `new_disease_epochs` on a fraction of one client's
data, and the new-disease experiment in Section 14 does the same, because both study data
efficiency rather than convergence at a fixed budget.

Every training run writes one row per client into `RESULTS`, tagged with experiment, setting and
method. The scalability sweep (Section 23) and reliability sweep (Section 24) add to that same
table, varying client count and seed. Section 25 reads back the FedMoE gate weights, Section 26
regenerates the parameter table from CONFIG and RESULTS, and Section 27 rebuilds every table and
writes `results.json`, so all reported numbers come from what actually executed.


---
## 27. Consolidated tables, regenerated from the full run

Section 17 runs before the experiments in Sections 19 to 26, so its tables cannot include them.
This section rebuilds every table from the final state of `RESULTS` and rewrites `results.json`,
so the saved file matches the notebook end to end.


In [ ]:
# Section 17 runs before the experiments added in Sections 19-26, so its tables cannot contain
# them. This cell regenerates everything from the final state of RESULTS and rewrites results.json.

show('TABLE 10 - fair sample-exposure protocol, e x R = E',
     [r for r in RESULTS if r.get('setting') == 'fair_protocol'],
     ['method', 'local_epochs', 'fair_rounds'], ['accuracy', 'f1', 'up_mb', 'time_s'])

show('TABLE 11 - FedMoE and FedMix against the main FedAvg run',
     [r for r in RESULTS if r.get('experiment') in ('fedmoe', 'fedmix')
      or (r.get('method') == 'FedAvg' and r.get('setting') == 'main')],
     ['method'], ['accuracy', 'f1', 'up_mb', 'time_s'], sort_by='accuracy')

show('TABLE 11b - FedMoE gate assignment per client',
     [r for r in RESULTS if r.get('experiment') == 'fedmoe'],
     ['method', 'client'], ['accuracy', 'f1', 'dominant_expert'])

show('TABLE 12 - LOCO bootstrap, averaged over the five held-out clients',
     rows(experiment='bootstrap_loco'), ['method', 'fraction'], ['accuracy', 'f1'])

show('TABLE 12b - LOCO bootstrap per held-out client',
     rows(experiment='bootstrap_loco'), ['client', 'fraction', 'method'], ['accuracy', 'f1'])

show('TABLE 13 - zero-shot transplant across seeds',
     rows(experiment='zero_shot_ablation'), ['client'], ['accuracy'])

show('TABLE 14 - scalability in the number of clients',
     [r for r in RESULTS if str(r.get('setting', '')).startswith('scalability')],
     ['setting'], ['accuracy', 'f1', 'up_mb', 'time_s'])

show('TABLE 15 - reliability across seeds (mean +- std over seeds, per client)',
     [r for r in RESULTS if str(r.get('setting', '')).startswith('reliability')],
     ['client'], ['accuracy', 'f1'])

show('TABLE 5b - privacy, re-run with the clipping grid placed around the measured update norm',
     rows(experiment='privacy'), ['dp_clip', 'dp_noise'], ['accuracy', 'f1', 'auc'])

show('TABLE 7a - Grad-CAM agreement (degenerate maps excluded)',
     rows(experiment='explainability'), ['method'], ['value', 'n_valid'])

show('TABLE 7b - linear probe, fit on val and scored on test',
     rows(experiment='linear_probe'), ['method', 'client'], ['accuracy'])

with open('results.json', 'w') as f:
    json.dump({'config': CONFIG, 'table1': TABLE1, 'results': RESULTS, 'history': HISTORY},
              f, indent=1, default=str)
print()
print('saved results.json -', len(RESULTS), 'result rows,', len(HISTORY), 'history rows')


---
## 28. The same campaign in a shared healthy/diseased label space (Q7)

**Question.** Everything above uses the 25-class superset, where each client owns a private block
of the output layer. That design answers "can one network name any condition of any modality",
but it makes two things impossible. A model pretrained on four clients has no usable head for the
fifth, so a genuine zero-shot bootstrap cannot be measured; and a client's accuracy cannot be
compared directly against another client's, because they are scored on different blocks.

This section collapses all five datasets to the same two classes, healthy and diseased, using the
mapping in `BINARY_MAPS`, and repeats the entire campaign: the two baselines, the three
weight-averaging algorithms, federated distillation, the mixture of experts at three expert
counts, FedMix, and the full leave-one-client-out bootstrap with its zero-shot point.

Two consequences are worth stating before the numbers arrive.

1. **The task is easier**, so every absolute number will be higher. The comparison of interest is
   therefore not binary against superset, but the *ordering* of the methods within each setting:
   if federation loses to local-only in both label spaces, that ordering is a property of the
   modality heterogeneity and not of the label design.
2. **Zero-shot transfer becomes measurable.** With one shared output space the head trained by
   four clients applies directly to the fifth, so the 0 % point of the bootstrap can be run with
   the head transplanted as well as the backbone. Section 22 could only leave the head random.


In [ ]:
def run_campaign(clients, tag, seed=0, expert_counts=(3,), top2_counts=(), lam=0.2,
                 include=('centralized', 'local_only', 'fedavg', 'fedprox', 'fedadam',
                          'feddistill', 'fedmoe', 'fedmix')):
    """Run the full set of methods on one set of clients under one protocol.

    Every method sees the same clients, the same images, the same output space and the same number
    of passes over the data, so that the only thing that differs between the rows is the algorithm.
    This is the function that makes 'repeat the same campaign' literal rather than a claim: the
    superset and binary runs call it with the same arguments and differ only in the clients."""
    out = {}
    start = len(RESULTS)

    def done(**crit):
        """True when this exact run is already in RESULTS. Every method in the campaign shares
        one `setting`, so the guard has to match on experiment and algorithm as well: without it
        a resumed campaign would append a second copy of each finished run and quietly shift the
        averages in the master table."""
        return any(all(r.get(k) == v for k, v in crit.items()) for r in RESULTS)

    def step(name, fn, **crit):
        if crit and done(**crit):
            print(f'{name} already present for setting {tag!r}, skipping')
            return None
        result = fn()
        tag_rows(start, campaign=tag)
        save_results()
        release(result)
        gpu_report('after release:')
        out[name] = result
        return result

    if 'centralized' in include:
        step('centralized', lambda: run_centralized(clients, seed=seed, setting=tag),
             experiment='centralized', setting=tag, seed=seed)
    if 'local_only' in include:
        step('local_only', lambda: run_local_only(clients, seed=seed, setting=tag),
             experiment='local_only', setting=tag, seed=seed)
    for algo in ('fedavg', 'fedprox', 'fedadam'):
        if algo in include:
            step(algo, lambda a=algo: run_federated(clients, algo=a, seed=seed, setting=tag),
                 experiment='federated', algo=algo, setting=tag, seed=seed)
    if 'feddistill' in include:
        step('feddistill', lambda: run_distillation(clients, seed=seed, setting=tag),
             experiment='federated', algo='feddistill', setting=tag, seed=seed)
    if 'fedmoe' in include:
        for K in expert_counts:
            step(f'fedmoe_K{K}',
                 lambda k=K: run_fedmoe(clients, num_experts=k, top_k=1, seed=seed,
                                        share_backbone=True, setting=f'{tag}_moe_K{k}'),
                 experiment='fedmoe', setting=f'{tag}_moe_K{K}', seed=seed)
        for K in top2_counts:
            step(f'fedmoe_K{K}_top2',
                 lambda k=K: run_fedmoe(clients, num_experts=k, top_k=2, seed=seed,
                                        share_backbone=True, setting=f'{tag}_moe_K{k}_top2'),
                 experiment='fedmoe', setting=f'{tag}_moe_K{K}_top2', seed=seed)
    if 'fedmix' in include:
        step('fedmix', lambda: run_fedmix(clients, lam=lam, seed=seed, setting=tag),
             experiment='fedmix', setting=tag, seed=seed)
    tag_rows(start, campaign=tag)
    return out

if RUN.get('binary_campaign', True):
    BIN_CLIENTS = make_clients(label_space='binary')
    print('binary label space | per-client classes:', BIN_CLIENTS['num_classes'],
          '| shared output space:', BIN_CLIENTS['total_classes'])
    for r in describe(BIN_CLIENTS):
        print('   ', r)
    print()

    BIN_RUNS = run_campaign(BIN_CLIENTS, 'binary', seed=CONFIG['seeds'][0],
                            expert_counts=CONFIG['expert_counts_binary'])
    save_results()
else:
    BIN_CLIENTS, BIN_RUNS = None, {}
    print('binary campaign not run')


---
## 29. Bootstrap and zero-shot in the shared label space (Q7, Q6)

The same leave-one-client-out protocol as Section 21 and the same three-arm zero-shot point as Section 22, now with a head that transfers. At the 0 %
point three arms are compared: ImageNet alone, the federated backbone with a random head, and the
federated backbone together with the federated head. The difference between the second and third
arm is the whole value of a shared output space, and it cannot be measured in the superset setting
at all.

Every number is read against the majority-class baseline printed for each client. A zero-shot
model that does not beat that baseline has not transferred anything, however high its accuracy
looks.


In [ ]:
if RUN.get('binary_campaign', True) and BIN_CLIENTS is not None:
    _start = len(RESULTS)
    BIN_LOCO_TABLE = loco_bootstrap(BIN_CLIENTS, fractions=CONFIG['bootstrap_fractions'],
                                    seed=CONFIG['seeds'][0], algo=CONFIG['bootstrap_algo'],
                                    setting='bootstrap_loco_binary')
    tag_rows(_start, campaign='binary')

    print()
    print('Zero-shot (0 % of the held-out client\'s data), binary label space')
    print(f"{'client':16s} {'majority':>9s} {'imagenet':>9s} {'fed-bb':>9s} {'fed-full':>9s}")
    for name in BIN_CLIENTS['names']:
        rws = [r for r in RESULTS if r.get('experiment') == 'bootstrap_loco'
               and r.get('campaign') == 'binary' and r.get('client') == name
               and r.get('fraction') == 0.0]
        get = lambda init: next((r['accuracy'] for r in rws if r.get('init') == init), float('nan'))
        print(f"{name:16s} {get('majority'):9.4f} {get('imagenet'):9.4f} "
              f"{get('federated-backbone'):9.4f} {get('federated-full'):9.4f}")

    print()
    print('Full bootstrap curve, binary label space')
    print(f"{'client':16s} {'frac':>5s} {'init':20s}  acc      f1")
    for row in BIN_LOCO_TABLE:
        print(f"{row['client']:16s} {row['fraction']:>5} {row['init']:20s}  "
              f"{row['accuracy']:.4f}  {row['f1']:.4f}")
    save_results()
else:
    print('binary bootstrap not run')


---
## 30. Master comparison: every method, every measure

One table per label space, carrying accuracy, macro F1 and AUC for **every** method that was run,
including the mixture of experts and FedMix. The earlier per-section tables report each experiment
in isolation; this is the single place where all of them are on one scale, which is what makes the
comparison readable.

Read every row against the noise floor from Section 24 before calling any ordering a result.


In [ ]:
def campaign_rows(campaign):
    keep = ('centralized', 'local_only', 'federated', 'fedmoe', 'fedmix')
    return [r for r in RESULTS
            if r.get('campaign') == campaign and r.get('experiment') in keep]

def master_table(campaign, title, quiet=False):
    rws = campaign_rows(campaign)
    if not quiet:
        print()
        print(title)
    if not rws:
        if not quiet:
            print('  no records - this campaign was not run')
        return []
    groups = {}
    for r in rws:
        groups.setdefault(r['method'], []).append(r)
    table = []
    for method, rs in groups.items():
        row = {'method': method, 'n_clients': len(rs)}
        for k in ('accuracy', 'f1', 'auc'):
            vals = [r[k] for r in rs if r.get(k) is not None]
            row[k] = float(np.mean(vals)) if vals else None
        mb = [r.get('up_mb', 0) + r.get('down_mb', 0) for r in rs if r.get('up_mb') is not None]
        row['total_mb'] = max(mb) if mb else 0.0
        row['experts_used'] = rs[0].get('experts_used')
        row['num_experts'] = rs[0].get('num_experts')
        table.append(row)
    table.sort(key=lambda r: (r['accuracy'] is None, -(r['accuracy'] or 0)))
    if quiet:
        return table
    print(f"{'method':38s} {'acc':>7s} {'F1':>7s} {'AUC':>7s} {'experts used':>13s} {'MB':>9s}")
    print('  ' + '-' * 84)
    for r in table:
        used = ('-' if r['experts_used'] is None
                else f"{r['experts_used']}/{r['num_experts']}")
        auc = '-' if r['auc'] is None else f"{r['auc']:7.4f}"
        print(f"{r['method']:38s} {r['accuracy']:7.4f} {r['f1']:7.4f} {auc:>7s} "
              f"{used:>13s} {r['total_mb']:9.1f}")
    return table

MASTER_UNION = master_table('union', 'MASTER TABLE A - superset label space (25 classes)')
MASTER_BINARY = master_table('binary', 'MASTER TABLE B - binary label space (healthy / diseased)')

print()
print('EXPERT SWEEP - how many experts are actually used (Q4)')
moe_rows = [r for r in RESULTS if r.get('experiment') == 'fedmoe']
seen = set()
if moe_rows:
    print(f"{'setting':24s} {'K':>3s} {'top_k':>6s} {'acc':>7s} {'F1':>7s} {'AUC':>7s} "
          f"{'used':>6s} {'dominant':>9s} {'entropy':>8s}")
    print('  ' + '-' * 86)
    for r in sorted(moe_rows, key=lambda r: (str(r.get('label_space')), r.get('num_experts', 0),
                                             r.get('top_k', 1))):
        s = r.get('setting')
        if s in seen:
            continue
        seen.add(s)
        same = [x for x in moe_rows if x.get('setting') == s]
        acc = mean_over_clients(same, 'accuracy')
        f1 = mean_over_clients(same, 'f1')
        auc = mean_over_clients(same, 'auc')
        print(f"{str(s):24s} {r.get('num_experts', 0):3d} {r.get('top_k', 1):6d} "
              f"{acc:7.4f} {f1:7.4f} {(auc if auc is not None else float('nan')):7.4f} "
              f"{r.get('experts_used', 0):>3d}/{r.get('num_experts', 0):<2d} "
              f"{r.get('distinct_dominant', 0):9d} {r.get('mean_gate_entropy', 0.0):8.3f}")
else:
    print('  no mixture-of-experts records')


---
## 31. Export for the report

Everything the written report needs, generated from `RESULTS` rather than retyped: `tables.tex`
with every table in booktabs form, `figs/*.pdf` with every figure as vector graphics, and
`results.csv` for any table that still has to be built by hand. The final cell packs them into
`report_assets.zip`, which uploads straight into an Overleaf project.

No number in the report is typed twice. If an experiment is re-run, this cell is re-run and the
report picks up the new values on the next compile.


In [ ]:
import csv
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 9, 'axes.grid': True, 'grid.color': '0.85',
    'grid.linewidth': 0.5, 'axes.axisbelow': True, 'figure.facecolor': 'white',
    'savefig.facecolor': 'white', 'pdf.fonttype': 42,
})
os.makedirs('figs', exist_ok=True)

def _f(v, nd=4):
    # Values can arrive as floats, ints, or the '--' placeholder used when a table is rebuilt
    # from results.json without re-running the data cell, so formatting has to tolerate all three.
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return '--'
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, (int, float)):
        return f'{float(v):.{nd}f}' if isinstance(v, float) else str(v)
    return str(v).replace('_', r'\_').replace('%', r'\%').replace('&', r'\&')

TEX_PARTS = {}

def tex_table(headers, body, caption, label, align=None, note=None):
    align = align or ('l' + 'r' * (len(headers) - 1))
    out = ['\\begin{table}[htbp]', '\\centering',
           f'\\caption{{{caption}}}', f'\\label{{{label}}}',
           '\\small', f'\\begin{{tabular}}{{{align}}}', '\\toprule',
           ' & '.join(f'\\textbf{{{h}}}' for h in headers) + ' \\\\', '\\midrule']
    for row in body:
        out.append(' & '.join(_f(c) for c in row) + ' \\\\')
    out += ['\\bottomrule', '\\end{tabular}']
    if note:
        out.append(f'\\\\[2pt]\\footnotesize {note}')
    out += ['\\end{table}', '']
    block = '\n'.join(out)
    # Each table is also written to its own file, so the report can place it where it belongs
    # with \input{tab_<name>} instead of pulling in everything at once.
    TEX_PARTS[label.replace('tab:', '')] = block
    return block

TEX = ['% Generated by the notebook. Do not edit by hand; re-run the export cell instead.', '']

# ---------------------------------------------------------------- datasets
TEX.append(tex_table(
    ['Client', 'Modality', 'Classes', 'Train', 'Val', 'Test', 'Imbalance'],
    [[r['client'].replace('MNIST', ''), r['modality'], r['classes'], r['train'], r['val'],
      r['test'], _f(r['imbalance'], 2)] for r in TABLE1],
    'The five clients. Imbalance is the largest class divided by the smallest.',
    'tab:datasets', align='llrrrrr'))

# ---------------------------------------------------------------- master tables
def master_body(campaign):
    body = []
    for r in master_table(campaign, '', quiet=True) or []:
        used = '--' if r['experts_used'] is None else f"{r['experts_used']}/{r['num_experts']}"
        body.append([r['method'], r['accuracy'], r['f1'], r['auc'], used,
                     f"{r['total_mb']:.0f}"])
    return body

for camp, cap, lab in [('union', 'All methods, superset label space (25 classes). Accuracy, '
                                 'macro F1 and AUC are averaged over the five clients.',
                        'tab:master-union'),
                       ('binary', 'All methods, binary label space (healthy vs diseased), same '
                                  'protocol and the same number of passes over the data.',
                        'tab:master-binary')]:
    b = master_body(camp)
    if b:
        TEX.append(tex_table(['Method', 'Accuracy', 'Macro F1', 'AUC', 'Experts used', 'MB'],
                             b, cap, lab))

# ---------------------------------------------------------------- per client, superset
pc = {}
for r in campaign_rows('union'):
    pc.setdefault(r['method'], {})[r['client']] = r['accuracy']
if pc:
    clients_order = CLIENTS['names']
    TEX.append(tex_table(
        ['Method'] + [c.replace('MNIST', '') for c in clients_order],
        [[m] + [v.get(c) for c in clients_order] for m, v in pc.items()],
        'Per-client accuracy in the superset label space. The average in '
        '\\autoref{tab:master-union} hides opposite effects on different clients.',
        'tab:perclient-union'))

# ---------------------------------------------------------------- expert sweep
moe = [r for r in RESULTS if r.get('experiment') == 'fedmoe']
if moe:
    rows_out, seen = [], set()
    for r in sorted(moe, key=lambda r: (str(r.get('label_space')), r.get('num_experts', 0),
                                        r.get('top_k', 1))):
        s = r.get('setting')
        if s in seen:
            continue
        seen.add(s)
        same = [x for x in moe if x.get('setting') == s]
        rows_out.append([r.get('label_space'), r.get('num_experts'), r.get('top_k'),
                         mean_over_clients(same, 'accuracy'), mean_over_clients(same, 'f1'),
                         mean_over_clients(same, 'auc'),
                         f"{r.get('experts_used')}/{r.get('num_experts')}",
                         r.get('distinct_dominant'), r.get('mean_gate_entropy')])
    TEX.append(tex_table(
        ['Space', '$K$', 'top-$k$', 'Accuracy', 'Macro F1', 'AUC', 'Used', 'Dominant', 'Entropy'],
        rows_out,
        'Mixture of experts as the number of experts grows. "Used" counts experts receiving at '
        'least 1\\% of the test images; "Dominant" counts distinct experts that are the first '
        'choice of at least one client.',
        'tab:experts', align='llrrrrrrr'))

# ---------------------------------------------------------------- bootstrap curves
def bootstrap_body(campaign):
    rws = [r for r in RESULTS if r.get('experiment') == 'bootstrap_loco'
           and r.get('campaign') == campaign and r.get('init') not in ('majority',)]
    if not rws:
        return []
    fr = sorted({r['fraction'] for r in rws if r['fraction'] > 0})
    body = []
    for name in sorted({r['client'] for r in rws}):
        for init in ('imagenet', 'federated'):
            line = [name.replace('MNIST', ''), init]
            for f in fr:
                v = next((r['accuracy'] for r in rws if r['client'] == name
                          and r['init'] == init and r['fraction'] == f), None)
                line.append(v)
            body.append(line)
    return body, fr

for camp, lab, cap in [('union', 'tab:bootstrap-union',
                        'Leave-one-client-out bootstrap, superset label space.'),
                       ('binary', 'tab:bootstrap-binary',
                        'Leave-one-client-out bootstrap, binary label space.')]:
    got = bootstrap_body(camp)
    if got:
        body, fr = got
        TEX.append(tex_table(['Held-out client', 'Start from'] + [f'{int(f*100)}\\%' for f in fr],
                             body,
                             cap + ' Each row is the accuracy reached after fine-tuning on that '
                                   'fraction of the held-out client\'s own training data.',
                             lab, align='ll' + 'r' * len(fr)))

# ---------------------------------------------------------------- zero shot
zs = [r for r in RESULTS if r.get('experiment') == 'bootstrap_loco' and r.get('fraction') == 0.0]
if zs:
    body = []
    for camp in ('union', 'binary'):
        for name in sorted({r['client'] for r in zs if r.get('campaign') == camp}):
            g = lambda init: next((r['accuracy'] for r in zs if r.get('campaign') == camp
                                   and r['client'] == name and r.get('init') == init), None)
            body.append([camp, name.replace('MNIST', ''), g('majority'), g('imagenet'),
                         g('federated-backbone'), g('federated-full')])
    if body:
        TEX.append(tex_table(
            ['Space', 'Client', 'Majority', 'ImageNet', 'Backbone only', 'Backbone + head'],
            body,
            'Accuracy with zero local training data. "Majority" is the constant predictor that '
            'always answers the most frequent test class, and is the floor any transferred model '
            'must beat. "Backbone + head" is only meaningful where all clients share one output '
            'space.',
            'tab:zeroshot', align='llrrrr'))

# ---------------------------------------------------------------- fair protocol
fair = [r for r in RESULTS if r.get('setting') == 'fair_protocol']
if fair:
    body, seen = [], set()
    for r in fair:
        k = (r.get('method'), r.get('local_epochs'), r.get('fair_rounds'))
        if k in seen:
            continue
        seen.add(k)
        same = [x for x in fair if (x.get('method'), x.get('local_epochs'),
                                    x.get('fair_rounds')) == k]
        e, R = r.get('local_epochs'), r.get('fair_rounds')
        body.append([f'{e} $\\times$ {R}' if e else 'Centralized',
                     mean_over_clients(same, 'accuracy'), mean_over_clients(same, 'f1'),
                     f"{(r.get('up_mb', 0) or 0) + (r.get('down_mb', 0) or 0):.0f}"])
    TEX.append(tex_table(['$e \\times R$', 'Accuracy', 'Macro F1', 'MB'], body,
                         'Matched sample exposure: every row performs the same total number of '
                         'passes over the data, $e \\cdot R = E = 32$.',
                         'tab:fair'))

# ---------------------------------------------------------------- privacy
pv = [r for r in RESULTS if r.get('experiment') == 'privacy']
if pv:
    body, seen = [], set()
    for r in pv:
        k = (r.get('dp_clip'), r.get('dp_noise'))
        if k in seen:
            continue
        seen.add(k)
        same = [x for x in pv if (x.get('dp_clip'), x.get('dp_noise')) == k]
        body.append([r.get('dp_clip'), r.get('dp_noise'), mean_over_clients(same, 'accuracy'),
                     mean_over_clients(same, 'f1'), mean_over_clients(same, 'auc')])
    body.sort(key=lambda b: (b[1] or 0, b[0] or 0))
    TEX.append(tex_table(['Clip $C$', 'Noise', 'Accuracy', 'Macro F1', 'AUC'], body,
                         'Cost of the update-level clipping and Gaussian noise mechanism. This is '
                         'a mechanism, not a formal $(\\varepsilon,\\delta)$ guarantee.',
                         'tab:privacy'))

# ---------------------------------------------------------------- scalability
sc = [r for r in RESULTS if str(r.get('setting', '')).startswith('scalability')]
if sc:
    body, seen = [], set()
    for r in sorted(sc, key=lambda r: str(r.get('setting'))):
        s = r.get('setting')
        if s in seen:
            continue
        seen.add(s)
        same = [x for x in sc if x.get('setting') == s]
        body.append([str(s).replace('scalability_n', ''), mean_over_clients(same, 'accuracy'),
                     mean_over_clients(same, 'f1'),
                     f"{(r.get('up_mb', 0) or 0) + (r.get('down_mb', 0) or 0):.0f}",
                     f"{r.get('time_s', 0):.0f}"])
    TEX.append(tex_table(['Clients', 'Accuracy', 'Macro F1', 'MB', 'Time (s)'], body,
                         'Behaviour as clients are added. Clients are always added in the same '
                         'order, so the number of clients and which clients took part are '
                         'confounded.', 'tab:scalability'))

# ---------------------------------------------------------------- reliability
rl = [r for r in RESULTS if str(r.get('setting', '')).startswith('reliability')]
if rl:
    body = []
    for name in CLIENTS['names']:
        vals = [r['accuracy'] for r in rl if r.get('client') == name]
        f1s = [r['f1'] for r in rl if r.get('client') == name]
        if vals:
            body.append([name.replace('MNIST', ''), float(np.mean(vals)), float(np.std(vals)),
                         float(np.mean(f1s)), float(np.std(f1s)), len(vals)])
    TEX.append(tex_table(['Client', 'Mean acc', 'Std acc', 'Mean F1', 'Std F1', 'Runs'], body,
                         'Run-to-run variation of FedAvg over repeated seeds. Any difference '
                         'smaller than the standard deviation is not a result.',
                         'tab:reliability'))

with open('tables.tex', 'w') as f:
    f.write('\n'.join(TEX))
for name, block in TEX_PARTS.items():
    with open(f'tab_{name}.tex', 'w') as f:
        f.write(block)
print(f'tables.tex written, plus {len(TEX_PARTS)} individual tab_*.tex files:',
      sorted(TEX_PARTS))

# ---------------------------------------------------------------- figures
def bar_methods(campaign, fname, title):
    rws = campaign_rows(campaign)
    if not rws:
        return False
    groups = {}
    for r in rws:
        groups.setdefault(r['method'], []).append(r)
    items = [(m, mean_over_clients(v, 'accuracy'), mean_over_clients(v, 'f1'))
             for m, v in groups.items()]
    items.sort(key=lambda t: -t[1])
    labels = [i[0].replace('FedMoE ', 'MoE ').replace('(lam=0.2)', '') for i in items]
    acc, f1 = [i[1] for i in items], [i[2] for i in items]
    x = np.arange(len(items))
    fig, ax = plt.subplots(figsize=(7.2, 3.2))
    ax.bar(x - 0.2, acc, 0.4, color='0.35', edgecolor='black', lw=0.6, label='Accuracy')
    ax.bar(x + 0.2, f1, 0.4, color='white', edgecolor='black', lw=0.6, hatch='///',
           label='Macro F1')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=7)
    ax.set_ylabel('Score')
    ax.set_title(title, fontsize=10)
    ax.legend(frameon=False, fontsize=8, ncol=2)
    fig.savefig(f'figs/{fname}.pdf', bbox_inches='tight')
    plt.close(fig)
    return True

bar_methods('union', 'main_union', 'Superset label space (25 classes)')
bar_methods('binary', 'main_binary', 'Binary label space (healthy vs diseased)')

if moe:
    seen, pts = set(), []
    for r in sorted(moe, key=lambda r: (r.get('num_experts', 0), r.get('top_k', 1))):
        s = r.get('setting')
        if s in seen or r.get('label_space') != 'union' or not r.get('share_backbone', True):
            continue
        seen.add(s)
        same = [x for x in moe if x.get('setting') == s]
        pts.append((r.get('num_experts'), r.get('top_k', 1), mean_over_clients(same, 'accuracy'),
                    r.get('experts_used', 0)))
    if pts:
        fig, ax = plt.subplots(figsize=(6.2, 3.0))
        for tk, mark, ls in [(1, 'o', '-'), (2, 's', '--')]:
            sel = [p for p in pts if p[1] == tk]
            if sel:
                ax.plot([p[0] for p in sel], [p[2] for p in sel], mark + ls, color='black',
                        lw=1.1, ms=4, label=f'top-{tk} routing')
        ax.set_xlabel('Number of experts $K$')
        ax.set_ylabel('Mean accuracy')
        ax.legend(frameon=False, fontsize=8)
        ax2 = ax.twinx()
        ax2.bar([p[0] for p in pts if p[1] == 1], [p[3] for p in pts if p[1] == 1], 0.45,
                color='white', edgecolor='0.5', lw=0.8, hatch='...')
        ax2.set_ylabel('Experts receiving $\\geq$1% of images')
        ax2.grid(False)
        fig.savefig('figs/experts.pdf', bbox_inches='tight')
        plt.close(fig)

def bootstrap_fig(campaign, fname):
    rws = [r for r in RESULTS if r.get('experiment') == 'bootstrap_loco'
           and r.get('campaign') == campaign and r.get('fraction', 0) > 0]
    if not rws:
        return
    fr = sorted({r['fraction'] for r in rws})
    names = sorted({r['client'] for r in rws})
    fig, ax = plt.subplots(figsize=(6.6, 3.2))
    for name, mark in zip(names, ['o', 's', '^', 'v', 'D']):
        gains = []
        for f in fr:
            a = next((r['accuracy'] for r in rws if r['client'] == name and r['fraction'] == f
                      and r['init'] == 'federated'), None)
            b = next((r['accuracy'] for r in rws if r['client'] == name and r['fraction'] == f
                      and r['init'] == 'imagenet'), None)
            gains.append(None if (a is None or b is None) else a - b)
        ax.plot([f * 100 for f in fr], gains, mark + '-', lw=1.0, ms=4,
                label=name.replace('MNIST', ''))
    ax.axhline(0, color='black', lw=0.9)
    ax.set_xlabel("Percentage of the held-out client's own data")
    ax.set_ylabel('Federated start minus\nImageNet start (accuracy)')
    ax.legend(frameon=False, fontsize=7, ncol=3)
    fig.savefig(f'figs/{fname}.pdf', bbox_inches='tight')
    plt.close(fig)

bootstrap_fig('union', 'bootstrap_union')
bootstrap_fig('binary', 'bootstrap_binary')

if zs:
    for camp in ('union', 'binary'):
        names = sorted({r['client'] for r in zs if r.get('campaign') == camp})
        if not names:
            continue
        arms = ['majority', 'imagenet', 'federated-backbone', 'federated-full']
        fig, ax = plt.subplots(figsize=(6.8, 3.0))
        w = 0.2
        for j, (arm, col, ht) in enumerate(zip(arms, ['white', '0.8', '0.5', '0.2'],
                                               ['xxx', '', '', ''])):
            vals = [next((r['accuracy'] for r in zs if r.get('campaign') == camp
                          and r['client'] == n and r.get('init') == arm), 0) for n in names]
            ax.bar(np.arange(len(names)) + (j - 1.5) * w, vals, w, color=col, hatch=ht,
                   edgecolor='black', lw=0.6, label=arm)
        ax.set_xticks(np.arange(len(names)))
        ax.set_xticklabels([n.replace('MNIST', '') for n in names], fontsize=8)
        ax.set_ylabel('Accuracy with no local data')
        ax.legend(frameon=False, fontsize=7, ncol=4)
        fig.savefig(f'figs/zeroshot_{camp}.pdf', bbox_inches='tight')
        plt.close(fig)

print('figures written:', sorted(os.listdir('figs')))

# ---------------------------------------------------------------- csv + zip
flat_keys = []
for r in RESULTS:
    for k in r:
        if k not in flat_keys and not isinstance(r[k], (list, dict)):
            flat_keys.append(k)
with open('results.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=flat_keys, extrasaction='ignore')
    w.writeheader()
    for r in RESULTS:
        w.writerow({k: v for k, v in r.items() if not isinstance(v, (list, dict))})
save_results()

import zipfile
with zipfile.ZipFile('report_assets.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('tables.tex')
    for name in TEX_PARTS:
        z.write(f'tab_{name}.tex')
    z.write('results.csv')
    z.write('results.json')
    for fn in sorted(os.listdir('figs')):
        z.write(os.path.join('figs', fn))
print('report_assets.zip written - upload this to Overleaf')


In [1]:
import torchvision
print(torchvision.__version__)

0.26.0+cu130
